# GPU × 5-Model Hosting Economics

Compute per-GPU hourly revenue, cost, profit, and breakeven utilisation for five LLMs (GLM-5.3-Flash, Kimi-K3, DeepSeek-V4.1-Flash, Tencent Hy4-preview, Xiaomi MiMo-V2.6-Flash) on eight GPU SKUs (B300, GB300, B200, GB200, H200, H100, MI355X, MI325X).

Prices: OpenRouter list prices or Moonshot published rate cards. Throughput: my B300 lab measurements (GLM only) or InferenceX public anchors.

**Open with the two main results below**; `[A1..A5]` appendices hold raw prices, lab data, and InferenceX provenance.


## 0 · Setup & data load

Bundles the CSVs shipped with the notebook, falling back to Google Drive links if a file is missing. Also normalises the Notion GPU-inventory export (some rows come in a raw 18-column schema that has to be mapped onto the canonical (option_id, gpu_class, listed_rate) tuple).


In [33]:
import os
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.width', 220)

try:
    from google.colab import drive                    # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# DATA SOURCES — each row: local filename · Google Drive file id · role
# (links come from the user's Drive, shared 2026-10-24)
DRIVE = {
    'my_gpu_opts.csv':('1yRqFNwWXHHuXD-C7hBgF-jE3acANhf12', 'Notion GPU Inventory Tracker'),
    'my_glm53_b300_benchmark.csv':('1O0n8QlKg-ZXIJTGvMyUzSJoejZ_zGH7e', 'my B300 lab'),
    'public_gpu_prices.csv':('1jkz0KFtzMXzlaj_0WnU1-IwYcpk2ghN4', 'public market scan'),
    'infx_glm53_anchors.csv':('1oHF_xReRZR_uKF3gYHVMyH_QDkeqnPo3', 'InferenceX anchors'),
    'infx_all_models_anchors.csv':('1P_aUGPblQKD62KHL6lrP0y9PdCced6By', 'InferenceX all-model anchors'),
    'grid_full_v2.csv':('1_a67j8pUKGKfB-QFeSsIPwyKWXrJkqmw', 'workbook grid'),
    'openrouter_top5_by_month.csv':('13uHoW9AueCpiUu2eiXhQtTGlIWZNxiYA', 'OpenRouter monthly'),
    'openrouter_weekly_top9_10-2025_to_10-2026.csv':('1OFRwAdXIc6BgVjbitpOnbYJK9I2VkuUJ', 'OpenRouter weekly'),
}
# Local-only: hosted gateway (prod netpreme) scan results, refreshed by hand from
# /Users/netprem_lee/work/llm-gateway-benchmarking. Not uploaded to Drive.

import os.path as _osp
_NOTEBOOK_DIR = _osp.dirname(_osp.abspath(__file__)) if '__file__' in globals() else os.getcwd()

def _local(name):
    """Return the first readable csv path. Checks notebook dir, /content (Colab),
    ./data, ./grid, ./* sibling/descendant dirs, and Drive. Robust to whichever
    cwd / unpack layout Colab or VS Code hands us."""
    candidates = []
    # standard cwd-based layout (matches Colab's upload of the zip)
    for base in ['.', 'data', 'grid', 'GPU-LLM-v2-Analytics-Colab']:
        candidates.append(os.path.join(base, name))
    # also check every direct subdirectory of cwd — Colab users may unpack
    # the zip into /content/<something>/ by a different name
    try:
        for sub in sorted(os.listdir('.')):
            if os.path.isdir(sub) and not sub.startswith('.'):
                candidates.append(os.path.join(sub, name))
    except Exception:
        pass
    # /content is where Colab puts uploaded files
    candidates.append(_osp.join('/content', name))
    # the notebook's own directory (when launched locally via VS Code)
    candidates.append(_osp.join(_NOTEBOOK_DIR, name))
    candidates.append(_osp.join(_NOTEBOOK_DIR, 'data', name))
    try:
        # Colab also mounts Drive if the user picks "upload from Drive"
        candidates.append(_osp.join('/content/drive/MyDrive', name))
    except Exception:
        pass
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

def load(name, required=True):
    """Load CSV: local copy → Drive .npz cache → Drive CSV (with Colab download)."""
    p = _local(name)
    if p:
        return pd.read_csv(p)
    if name not in DRIVE:
        if required: raise FileNotFoundError(name)
        return None
    fid, role = DRIVE[name]
    url = f'https://drive.google.com/uc?id={fid}'
    try:
        return pd.read_csv(url, engine='python')
    except Exception as e:
        if required:
            raise RuntimeError(f'cannot load {name} from Drive ({e}). Open '
                               f'https://drive.google.com/file/d/{fid}/view and share to Anyone with the link.')
        return None

# if my_gpu_opts.csv came from the Notion tracker (raw 18-col export), map to grid schema
def _notion_to_myquotes(raw):
    import re as _re
    GPU_MAP = {'GB300':'gb300','B300':'b300','B200':'b200','GB200':'gb200',
               'H200':'h200','H100':'h100','MI355X':'mi355x','MI325X':'mi325x'}
    def _term(m):
        if pd.isna(m): return ('on_demand',0)
        m=int(m)
        if m<=1: return ('on_demand',0)
        if m<=13: return ('1_year',12)
        if m<=27: return ('2_year',24)
        if m<=44: return ('3_year',36)
        if m<=70: return ('5_year',60)
        return (f'{m}m',0)
    out=[]
    for _,r in raw.iterrows():
        gt = str(r.get('GPU Type','')).strip().upper()
        if gt not in GPU_MAP: continue
        g = GPU_MAP[gt]
        rt,tm = _term(r.get('Contract Length (month)'))
        rate = r.get('Unit Price')
        if pd.isna(rate): continue
        delivery = str(r.get('Delivery / RFS','')) if pd.notna(r.get('Delivery / RFS')) else ''
        delivery = _re.sub(r'\s+',' ',delivery)[:200]
        card = str(r.get('Quote Card','')).strip()
        upd  = str(r.get('Last Updated','')).strip() if pd.notna(r.get('Last Updated')) else ''
        try: price_date = pd.to_datetime(upd).date().isoformat()
        except: price_date=''
        avail = str(r.get('Availability Signal','')).strip()
        if avail == 'Inactive Quote': continue
        qc_short = _re.sub(r'https?://','',card)[:34].replace(' ','-')
        out.append(dict(
            option_id       = f"{g.upper()}-{rt.replace('_','')}-{qc_short}"[:52],
            gpu_class       = g,
            delivery_rfs    = delivery,
            availability_signal = avail,
            rental_term     = rt,
            term_months     = tm,
            order_size_gpus = np.nan,
            listed_rate     = round(float(rate),3),
            upfront         = np.nan,
            upfront_pct     = float(r['Prepayment']) if pd.notna(r.get('Prepayment')) else np.nan,
            eff_cost_ph     = round(float(rate),3),
            breakeven_tps_sheet = np.nan,
            source          = (str(r['Provider/Aggregator'])+' · Notion tracker') if pd.notna(r.get('Provider/Aggregator')) else 'Notion tracker',
            source_url      = 'https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559',
            upfront_kind    = 'prepayment',
            source_type     = 'my_quotes',
            price_date      = price_date,
            upfront_per_hour= np.nan,
        ))
    return pd.DataFrame(out)

def _ensure_myquotes(df):
    """Return df already in grid schema. If it's the raw Notion tracker (has 'Quote Card'), map it."""
    if df is None: return None
    if 'Quote Card' in df.columns:
        return _notion_to_myquotes(df)
    return df

# ── Colab fallback: if we got here but haven't actually unpacked the bundle,
# try to find GPU-LLM-v2-Analytics-Colab.zip in cwd and unzip it in place.
# This handles the common Colab flow of "upload the zip, then open the
# notebook from it" — Colab doesn't auto-extract archives.
import zipfile as _zipfile
if _local('my_gpu_opts.csv') is None:
    # try every known zip naming pattern in cwd
    for _zp in ['GPU-LLM-v2-Analytics-Colab.zip', 'colab_upload.zip']:
        if os.path.exists(_zp):
            with _zipfile.ZipFile(_zp) as _z:
                _z.extractall('.')
            print(f"unzipped {_zp} into cwd")
            break

my_quotes = _ensure_myquotes(load('my_gpu_opts.csv', required=True))
my_bench  = load('my_glm53_b300_benchmark.csv',         required=True)

def opt(n):  # optional tables — notebook tolerates absence
    try:    return load(n, required=False)
    except: return None

pub_prices   = opt('public_gpu_prices.csv')
infx_glm53   = opt('infx_glm53_anchors.csv')
infx_all_models = opt('infx_all_models_anchors.csv')
my_gw_bench    = opt('my_glm53_gateway_bench.csv')  # hosted-service perf (prod 2026-11-09 c64 scan)
if infx_all_models is None:
    # Re-compute every candidate the same way _local does and report what it tried
    # (the diagnostic above only listed the static '4 base paths' — the new code
    # also scans cwd subdirs and /content directly, so we want to print every
    # path actually probed).
    _probe_paths = (
        [os.path.join(base, 'infx_all_models_anchors.csv') for base in
         ['.', 'data', 'grid', 'GPU-LLM-v2-Analytics-Colab']]
        + [os.path.join(sub, 'infx_all_models_anchors.csv') for sub in
           sorted(s for s in os.listdir('.')
                  if os.path.isdir(s) and not s.startswith('.'))]
        + ['/content/infx_all_models_anchors.csv',
           os.path.join(_NOTEBOOK_DIR, 'infx_all_models_anchors.csv'),
           os.path.join(_NOTEBOOK_DIR, 'data', 'infx_all_models_anchors.csv'),
           '/content/drive/MyDrive/infx_all_models_anchors.csv']
    )
    print(f"  ⚠ infx_all_models_anchors.csv not found in any of these {len(_probe_paths)} paths:")
    for p_ in _probe_paths:
        print(f"      tried: {p_}  (exists={os.path.exists(p_)})")
    print(f"  cwd={os.getcwd()}; notebook_dir={_NOTEBOOK_DIR}")
    print("  hint: run `!ls -la /content` in a Colab cell to see where your bundle actually is")
grid_full    = opt('grid_full_v2.csv')
or_monthly   = opt('openrouter_top5_by_month.csv')
or_weekly    = opt('openrouter_weekly_top9_10-2025_to_10-2026.csv')

print('loaded:',
      f'my_quotes={len(my_quotes)}', f'my_bench={len(my_bench)}',
      f'public={0 if pub_prices is None else len(pub_prices)}',
      f'infx={0 if infx_glm53 is None else len(infx_glm53)}',
      f'infx_all={0 if infx_all_models is None else len(infx_all_models)}',
      f'my_gw={0 if my_gw_bench is None else len(my_gw_bench)}',
      f'grid={0 if grid_full is None else len(grid_full)}',
      f'or_monthly={0 if or_monthly is None else len(or_monthly)}',
      f'or_weekly={0 if or_weekly is None else or_weekly.shape[1]}')

# ── regression-test pack ──────────────────────────────
# tests/test_cells.json pins a handful of cell-5/cell-7 outputs to known values.
# tests/run_regression.py recomputes them and reports PASS/FAIL.
# Here we just note that the pack was found, so notebook users see the pointer.
import os as _os, json as _json
_TEST_CANDS = [_os.path.join(_NOTEBOOK_DIR, 'tests', 'test_cells.json'),
               _os.path.join('.', 'tests', 'test_cells.json'),
               _os.path.join('/content', 'tests', 'test_cells.json'),
               _os.path.join('GPU-LLM-v2-Analytics-Colab', 'tests', 'test_cells.json')]
_test_path = next((c for c in _TEST_CANDS if _os.path.exists(c)), None)
if _test_path:
    try:
        _pack = _json.load(open(_test_path))
        print(f'regression pack · {len(_pack.get("tests",[]))} pinned cases · see {os.path.basename(_test_path)}')
    except Exception as _e:
        print(f'regression pack · parse fail: {_e}')
else:
    print('regression pack · tests/test_cells.json not found beside notebook (ok if you only shipped the .ipynb)')



loaded: my_quotes=66 my_bench=30 public=52 infx=21 infx_all=68 grid=2300 or_monthly=26 or_weekly=54
regression pack · 4 pinned cases · see test_cells.json


## 0b · Fleet & leasing overview — what we're actually spending each month

Interactive panel over the live GPU-lease sheet (Google-Drive CSV mirror of your Notion tracker). The cell's inputs sit at the top:

- **Layout dropdown** — flip between `column` (KPI strip, then all charts stacked) and `tabs` (KPI strip + one tab per chart). Try both; we'll keep whichever reads better.
- **Occupancy sliders** — one per Internal host. Drag to update; fires on release (no render storm mid-drag).

Data / money inputs (top of the code cell):

- `FLEET_DEFAULT_OCCUPANCY` — slider starting values; edit to make a current state "stick" across runs.
- `FLEET_CUSTOMER_MONTHLY_REVENUE` — $/month actually collected per customer. `{}` = unknown → revenue KPI shows "—" and the P&L waterfall is cost-only.
- `FLEET_DRIVE_URL` — bump `&v=` after editing the sheet to bypass Colab/Drive HTTP cache.

Rendering detail: charts are generated as PNG bytes into `W.Image` widgets and the result area is replaced atomically on every change — no duplicate printing, no stacked outputs.


In [34]:
# ── Section 0b · Fleet & Cost panel ─────────────────────────────────────────
# Layout dropdown at the top switches between 'column' and 'tabs' demos.
# Sliders drive per-host occupancy; every change replaces the result area
# atomically (no re-print stacking). Figs are PNG bytes in W.Image widgets —
# no plt.show, no Output widget, no ghost duplicates.

import io
from datetime import datetime
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display, HTML

try:
    import ipywidgets as W
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False

# ── user inputs ─────────────────────────────────────────────────────────────
FLEET_DEFAULT_OCCUPANCY = {
    "Reactor (Alibaba)  B300 - 47.250.177.205": 8,
    "Reactor (Alibaba) B300 - 47.250.166.190":  8,
    "Manifold B300 - 63.97.218.1":              8,
    "Manifold B300 - 63.97.218.8":              8,
    "Vessl H100":                               0,
}
FLEET_CUSTOMER_MONTHLY_REVENUE = {}   # {"PhysionLabs": 85000}
FLEET_DRIVE_URL = ("https://drive.google.com/uc?export=download"
                   "&id=1OM-q_CKbJBhbm_ShtUIMd5HFP7WaIpWo&v=2026-11-04")

# ── load & enrich ─────────────────────────────────────────────────────────
def _fleet_load():
    try:    return pd.read_csv(FLEET_DRIVE_URL, encoding="utf-8-sig")
    except Exception:
        return pd.read_csv("fleet_gpu_rentals.csv", encoding="utf-8-sig")

_raw = _fleet_load()
_base = _raw[pd.to_numeric(_raw["GPU Count"], errors="coerce").notna()].copy()
_base["GPU Count"]    = _base["GPU Count"].astype(float).astype(int)
_base["Unit Price"]   = pd.to_numeric(_base["Unit Price ($/GPU/h)"], errors="coerce")
_base["Monthly Cost"] = pd.to_numeric(_base["Monthly Cost ($/month)"].astype(str).str.replace(",", ""), errors="coerce")
_base["End Date"]     = pd.to_datetime(_base["End Date"], errors="coerce")
for _c in ("Resource", "Vendor", "GPU Model", "Allocation Scope", "Purpose"):
    _base[_c] = _base[_c].astype(str).str.strip()
_base["Customer"] = _base["Customer"].astype(str).str.strip().replace("-", "")
_base = (_base.sort_values(["Allocation Scope", "GPU Model", "Vendor"],
                           ascending=[False, True, True]).reset_index(drop=True))
_INTERNAL_RESOURCES = _base.loc[_base["Allocation Scope"] == "Internal", "Resource"].tolist()

def _default_occ(res):
    for k, v in FLEET_DEFAULT_OCCUPANCY.items():
        if res.strip().startswith(k.strip()[:30]):
            return v
    return None

def _compute(occ_overrides=None, rev_overrides=None):
    f = _base.copy()
    occ = {}
    for res in f["Resource"]:
        if res in _INTERNAL_RESOURCES:
            occ[res] = (occ_overrides or {}).get(res, _default_occ(res))
    f["GPUs In Use"] = f["Resource"].map(occ).fillna(f["GPU Count"])
    f["GPUs In Use"] = f[["GPUs In Use", "GPU Count"]].min(axis=1).astype(int)
    f["Occupancy %"] = np.where(f["GPU Count"] > 0,
                                100.0 * f["GPUs In Use"] / f["GPU Count"], np.nan)
    _rev_map = {**FLEET_CUSTOMER_MONTHLY_REVENUE, **(rev_overrides or {})}
    f["Customer Monthly Revenue"] = pd.to_numeric(
        f["Customer"].map(_rev_map), errors="coerce")
    f["Net Contribution"] = f["Customer Monthly Revenue"].fillna(0.0) - f["Monthly Cost"]
    return f

def _totals(f):
    i = f[f["Allocation Scope"] == "Internal"]; c = f[f["Allocation Scope"] == "Customer"]
    t = dict(
        total_monthly_cost    = float(f["Monthly Cost"].sum()),
        internal_monthly_cost = float(i["Monthly Cost"].sum()),
        customer_monthly_cost = float(c["Monthly Cost"].sum()),
        total_gpus            = int(f["GPU Count"].sum()),
        internal_gpus         = int(i["GPU Count"].sum()),
        customer_gpus         = int(c["GPU Count"].sum()),
        internal_in_use       = int(i["GPUs In Use"].sum()),
        customer_revenue      = float(f["Customer Monthly Revenue"].sum()),
        net_pnl               = float(f["Net Contribution"].sum()),
    )
    t["internal_occupancy"] = 100.0 * t["internal_in_use"] / max(1, t["internal_gpus"])
    t["known_revenue"]      = t["customer_revenue"] > 0
    return t

# ── renderers (pure: return bytes / html strings) ─────────────────────────
def _fig_cost_bytes(f):
    tbl = (f.groupby(["GPU Model", "Vendor", "Allocation Scope"], observed=True)["Monthly Cost"]
            .sum().unstack(fill_value=0).reindex(columns=["Internal", "Customer"], fill_value=0))
    fig, ax = plt.subplots(figsize=(9, 4.4), dpi=110)
    xs = np.arange(len(tbl)); w = 0.38
    b1 = ax.bar(xs - w/2, tbl["Internal"], width=w, color="#64748b", label="Internal (R&D / self-use)")
    b2 = ax.bar(xs + w/2, tbl["Customer"], width=w, color="#10b981", label="Customer (rented out)")
    ax.bar_label(b1, fmt=lambda v: f"${v:,.0f}" if v > 0 else "", fontsize=8, color="#0f172a")
    ax.bar_label(b2, fmt=lambda v: f"${v:,.0f}" if v > 0 else "", fontsize=8, color="#0f172a")
    ax.set_xticks(xs); ax.set_xticklabels([f"{m}\n{v}" for m, v in tbl.index], fontsize=9)
    ax.set_ylabel("Monthly cost ($)"); ax.set_title("Fleet monthly cost — by GPU model × vendor × scope")
    ax.yaxis.set_major_formatter(lambda v, p: f"${v/1000:.0f}k")
    ax.grid(axis="y", alpha=0.25); ax.legend(loc="upper right", fontsize=9, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", bbox_inches="tight"); plt.close(fig)
    return buf.getvalue()

def _fig_occupancy_bytes(f):
    """Per-GPU waffle: one square = one GPU, filled = in use. Right-hand stat
    columns carry used/total, occ%, $/mo, idle $/mo, expiry + days left.

    Layout is normalised to a 0–100 x-range so the rendered PNG stays ~9.6 in
    wide (readable when displayed at the panel's 920px)."""
    import textwrap
    from matplotlib.lines import Line2D
    from matplotlib.patches import Rectangle

    df = f[f["Allocation Scope"] == "Internal"].sort_values("Monthly Cost", ascending=False)
    today = pd.Timestamp(datetime.now().date())
    n     = len(df)

    # figure sized for the 920px panel: 9.6in @150dpi ≈ 1440px wide
    fig, ax = plt.subplots(figsize=(9.6, 0.62 * n + 1.1), dpi=150)
    ax.set_xlim(0, 100)
    ax.set_ylim(-0.55, n + 1.30)
    ax.axis("off")

    LBL_X  =  0.0
    TILE_X = 31.0          # first tile left edge
    TILE_W =  2.7          # tile width
    TILE_G =  0.5          # gap between tiles
    S_USED, S_PCT, S_COST, S_IDLE, S_EXP = 63.5, 70.5, 76.0, 83.5, 91.0

    def _band(p):
        if p >= 75: return "#059669"
        if p >= 40: return "#d97706"
        if p >  0:  return "#dc2626"
        return "#94a3b8"

    # title + column headers
    ax.text(LBL_X, n + 0.92, "Internal fleet — one square = one GPU (coloured = in use)",
            fontsize=10.5, fontweight="bold", color="#0f172a", va="center")
    for x, hdr, ha in [(S_USED, "used / total", "left"), (S_PCT, "occ%", "left"),
                       (S_COST, "cost / mo", "left"), (S_IDLE, "idle $ / mo", "left"),
                       (S_EXP,  "expires · days left", "left")]:
        ax.text(x, n + 0.30, hdr, fontsize=7.4, color="#64748b", style="italic", ha=ha)

    for yi, (_, r) in enumerate(df.iterrows()):
        y = n - 1 - yi                      # row baseline (top row highest)
        cnt, used = int(r["GPU Count"]), int(r["GPUs In Use"])
        pct  = float(r["Occupancy %"])
        idle = float(r["Monthly Cost"]) * (1 - pct / 100.0)
        col  = _band(pct)

        # host label, wrapped to two lines, right-aligned up to the tile zone
        lbl = "\n".join(textwrap.wrap(str(r["Resource"]).strip(), 30)[:2])
        ax.text(TILE_X - 1.6, y + 0.5, lbl, ha="right", va="center",
                fontsize=8.2, color="#0f172a")

        # one square per GPU
        for k in range(cnt):
            x = TILE_X + k * (TILE_W + TILE_G)
            face = col if k < used else "#f1f5f9"
            edge = col if k < used else "#cbd5e1"
            ax.add_patch(Rectangle((x, y + 0.17), TILE_W, 0.66,
                                   facecolor=face, edgecolor=edge, lw=0.8))

        # stat columns
        end  = r["End Date"]
        days = (end - today).days if pd.notna(end) else None
        end_s = end.strftime("%Y-%m-%d") if pd.notna(end) else "—"
        d_s   = f"{int(days)}d" if days is not None else "—"
        exp_col = "#dc2626" if (days is not None and days <= 180) else "#475569"
        idle_col = "#dc2626" if idle > 5000 else "#64748b"

        ax.text(S_USED, y + 0.5, f"{used}/{cnt}", fontsize=9, fontweight="bold",
                color="#0f172a", va="center")
        ax.text(S_PCT,  y + 0.5, f"{pct:4.0f}%", fontsize=9, fontweight="bold",
                color=col, va="center")
        ax.text(S_COST, y + 0.5, f"${r['Monthly Cost']:,.0f}", fontsize=8.4,
                color="#0f172a", va="center")
        ax.text(S_IDLE, y + 0.5, (f"${idle:,.0f}" if idle > 0 else "—"), fontsize=8.4,
                color=idle_col, fontweight=("bold" if idle > 5000 else "normal"), va="center")
        ax.text(S_EXP,  y + 0.5, f"{end_s} · {d_s}", fontsize=8.4,
                color=exp_col, va="center")

        # hairline between hosts
        if yi < n - 1:
            ax.axhline(y, color="#f1f5f9", lw=0.8, zorder=0)

    leg = [Line2D([0],[0], marker="s", color="none", markerfacecolor="#059669",
                  markeredgecolor="#059669", markersize=9, label="in use · occ ≥75%"),
           Line2D([0],[0], marker="s", color="none", markerfacecolor="#d97706",
                  markeredgecolor="#d97706", markersize=9, label="in use · 40–75%"),
           Line2D([0],[0], marker="s", color="none", markerfacecolor="#dc2626",
                  markeredgecolor="#dc2626", markersize=9, label="in use · <40%"),
           Line2D([0],[0], marker="s", color="none", markerfacecolor="#f1f5f9",
                  markeredgecolor="#cbd5e1", markersize=9, label="idle GPU")]
    ax.legend(handles=leg, loc="lower left", fontsize=7.6, frameon=False, ncol=4,
              bbox_to_anchor=(0.0, -0.14), handlelength=1.4, columnspacing=1.6)

    plt.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", bbox_inches="tight"); plt.close(fig)
    return buf.getvalue()

def _fig_pnl_bytes(t):
    """Three bars: Money in (revenue) / Money out (stacked fleet cost) / Net
    (plotted below the zero line when negative). Nothing overlaps — the net
    bar hangs under the axis and the legend sits in the empty top-left."""
    rev = t["customer_revenue"] if t["known_revenue"] else 0.0
    cost_ext = t["customer_monthly_cost"]
    cost_int = t["internal_monthly_cost"]
    net  = t["net_pnl"]
    cost_total = cost_ext + cost_int

    from matplotlib.lines import Line2D
    fig, ax = plt.subplots(figsize=(7.8, 4.6), dpi=150)
    top = max(rev, cost_total, abs(net), 1)

    X_IN, X_OUT, X_NET = 0, 1, 2

    # ── Money in ──
    ax.bar(X_IN, rev, color="#10b981", width=0.56)
    ax.text(X_IN, rev + top * 0.02, f"${rev:,.0f}" if rev else "$0",
            ha="center", fontsize=11, fontweight="bold", color="#047857")
    if rev == 0:
        ax.text(X_IN, top * 0.42, "no customer\nrevenue yet", ha="center",
                fontsize=9.5, color="#94a3b8", style="italic", va="center")

    # ── Money out (stacked) ──
    ax.bar(X_OUT, cost_int, color="#ef4444", width=0.56)
    ax.bar(X_OUT, cost_ext, bottom=cost_int, color="#f97316", width=0.56)
    if cost_int > 0:
        ax.text(X_OUT, cost_int / 2, f"${cost_int:,.0f}", ha="center", va="center",
                fontsize=9, color="white", fontweight="bold")
    if cost_ext > 0:
        ax.text(X_OUT, cost_int + cost_ext / 2, f"${cost_ext:,.0f}", ha="center",
                va="center", fontsize=9, color="white", fontweight="bold")
    ax.text(X_OUT, cost_total + top * 0.02, f"${cost_total:,.0f}",
            ha="center", fontsize=11, fontweight="bold", color="#b91c1c")

    # ── Net (hangs below zero when negative) ──
    net_color = "#059669" if net >= 0 else "#b91c1c"
    ax.bar(X_NET, net, color=net_color, width=0.56, alpha=0.88)
    ax.text(X_NET, net + (top * 0.02 if net >= 0 else -top * 0.02), f"${net:+,.0f}",
            ha="center", va="bottom" if net >= 0 else "top",
            fontsize=11, fontweight="bold", color=net_color)

    ax.axhline(0, color="#0f172a", lw=1.1)
    ax.set_xticks([X_IN, X_OUT, X_NET])
    ax.set_xticklabels(["Money in\n(customer revenue)", "Money out\n(fleet cost)",
                        "Net\nper month"], fontsize=10)
    ax.set_ylim(-top * 1.16, top * 1.16)
    ax.yaxis.set_major_formatter(lambda v, _p: f"${abs(v)/1000:.0f}k")
    ax.grid(axis="y", alpha=0.22)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)

    leg = [Line2D([0], [0], marker="s", color="none", markerfacecolor="#10b981",
                  markersize=10, label="customer revenue"),
           Line2D([0], [0], marker="s", color="none", markerfacecolor="#ef4444",
                  markersize=10, label="internal cost (R&D / self-use)"),
           Line2D([0], [0], marker="s", color="none", markerfacecolor="#f97316",
                  markersize=10, label="customer cost (rented out)")]
    ax.legend(handles=leg, loc="upper left", fontsize=8.6, frameon=False)

    cov = (rev / cost_total * 100) if cost_total else 0
    sub = (f"customer revenue covers {cov:.0f}% of the monthly fleet bill"
           if t["known_revenue"] else
           "add a row to FLEET_CUSTOMER_MONTHLY_REVENUE to see coverage")
    ax.text(0.5, -0.14, sub, transform=ax.transAxes, ha="center",
            fontsize=9, color="#475569")

    ax.set_title("Monthly P&L — money in vs money out vs net",
                 fontsize=11.5, fontweight="bold", color="#0f172a", pad=10)
    plt.tight_layout()
    buf = io.BytesIO(); fig.savefig(buf, format="png", bbox_inches="tight"); plt.close(fig)
    return buf.getvalue()

def _expiry_table_html(f):
    """Per-host expiry table — date, days-left badge, GPUs freed, $/mo freed,
    scope. Rows expiring within 6 mo get a red highlight."""
    today = pd.Timestamp(datetime.now().date())
    dd = (f.dropna(subset=["End Date"])
           .loc[:, ["Resource", "Allocation Scope", "GPU Model", "GPU Count",
                    "Monthly Cost", "End Date"]]
           .copy())
    dd["days"] = (dd["End Date"] - today).dt.days
    dd = dd.sort_values("End Date").reset_index(drop=True)

    def _badge(days):
        if days is None or pd.isna(days):   return '<span style="color:#94a3b8;">—</span>'
        if days <    0: return f'<span style="background:#7f1d1d; color:#fff; padding:1px 7px; border-radius:4px; font-weight:600;" title="ends {int(abs(days))}d ago — check if still being billed">ended {-int(days)} d ago</span>'
        if days <=  90: return f'<span style="background:#dc2626; color:#fff; padding:1px 7px; border-radius:4px; font-weight:600;">{int(days)} d</span>'
        if days <= 180: return f'<span style="background:#f59e0b; color:#fff; padding:1px 7px; border-radius:4px; font-weight:600;">{int(days)} d</span>'
        if days <= 365: return f'<span style="color:#0f172a; font-weight:600;">{int(days)} d</span>'
        return f'<span style="color:#64748b;">{int(days)} d</span>'

    rows_html = []
    for _, r in dd.iterrows():
        hl = ' style="background:#fef2f2;"' if (pd.notna(r["days"]) and r["days"] <= 180) else ""
        rows_html.append(
            f"<tr{hl}>"
            f'<td style="text-align:left;">{r["Resource"]}</td>'
            f"<td>{r['Allocation Scope']}</td>"
            f"<td>{r['GPU Model']}</td>"
            f"<td>{int(r['GPU Count'])}</td>"
            f"<td>${r['Monthly Cost']:,.0f}</td>"
            f"<td>{r['End Date'].strftime('%Y-%m-%d')}</td>"
            f"<td>{_badge(r['days'])}</td>"
            f"</tr>")

    tbl = (
        '<div style="border:1px solid #e2e8f0; border-radius:6px; overflow:hidden;">'
        '<table style="width:100%; border-collapse:collapse; font:12px/1.45 system-ui;">'
        '<thead><tr style="background:#f8fafc; color:#475569; font-size:11px; text-transform:uppercase; letter-spacing:.05em;">'
        '<th style="text-align:left; padding:6px 10px;">Host</th>'
        '<th style="padding:6px 10px;">Scope</th>'
        '<th style="padding:6px 10px;">GPU</th>'
        '<th style="padding:6px 10px;">Count</th>'
        '<th style="padding:6px 10px;">$/mo freed</th>'
        '<th style="padding:6px 10px;">End date</th>'
        '<th style="padding:6px 10px;">Days left</th>'
        "</tr></thead><tbody>"
        + "".join(rows_html) +
        '</tbody></table></div>'
        '<style>td{padding:5px 10px; text-align:center; border-top:1px solid #f1f5f9;}</style>'
    )
    return tbl

def _kpi_html(t):
    def card(label, value, sub="", color="#0f172a"):
        return (f'<div style="flex:1; min-width:170px; padding:14px 16px; border-radius:10px; '
                f'background:#f8fafc; border:1px solid #e2e8f0;">'
                f'<div style="font:600 10px/1.2 system-ui; letter-spacing:.08em; color:#64748b; '
                f'text-transform:uppercase;">{label}</div>'
                f'<div style="font:700 22px/1.2 system-ui; color:{color}; margin-top:4px;">{value}</div>'
                f'<div style="font:400 11px/1.3 system-ui; color:#64748b; margin-top:3px;">{sub}</div></div>')
    net_color = "#b91c1c" if t["net_pnl"] < 0 else "#059669"
    cov = (f'{t["customer_revenue"]/t["internal_monthly_cost"]*100:.0f}% of Internal cost'
           if t["known_revenue"] and t["internal_monthly_cost"] > 0 else "no revenue data yet")
    rev = f'${t["customer_revenue"]:,.0f}' if t["known_revenue"] else "—"
    return ('<div style="display:flex; flex-wrap:wrap; gap:10px; margin:8px 0 14px;">'
            + card("Total GPUs", f'{t["total_gpus"]}',
                   f'Internal {t["internal_gpus"]} · Customer {t["customer_gpus"]}')
            + card("Total monthly cost", f'${t["total_monthly_cost"]:,.0f}',
                   f'Internal ${t["internal_monthly_cost"]:,.0f} · Customer ${t["customer_monthly_cost"]:,.0f}')
            + card("Customer revenue", rev, cov)
            + card("Net P&amp;L / month", f'${t["net_pnl"]:+,.0f}', "revenue − all fleet cost", net_color)
            + card("Internal occupancy", f'{t["internal_occupancy"]:.0f}%',
                   f'{t["internal_in_use"]}/{t["internal_gpus"]} GPUs in use')
            + '</div>')

def _table_html(f):
    cols = ["Resource", "Vendor", "GPU Model", "Allocation Scope", "Customer",
            "GPU Count", "GPUs In Use", "Occupancy %", "Unit Price", "Monthly Cost",
            "Customer Monthly Revenue", "Net Contribution", "End Date", "Purpose"]
    tbl = (f[cols].assign(**{"End Date": f["End Date"].dt.strftime("%Y-%m-%d")})
           .style
           .format({"Unit Price": "${:.2f}", "Monthly Cost": "${:,.0f}",
                    "Customer Monthly Revenue": lambda v: f"${v:,.0f}" if pd.notna(v) else "—",
                    "Net Contribution": lambda v: f"${v:+,.0f}",
                    "Occupancy %": "{:.0f}%"})
           .bar(subset=["Occupancy %"], color="#0ea5e9", vmin=0, vmax=100)
           .set_properties(**{"font-size": "12px", "text-align": "left"}))
    def _row(r):
        if r["Allocation Scope"] == "Customer":
            return ["background-color:#ecfdf5"] * len(r)
        if r["Occupancy %"] == 0:
            return ["background-color:#fef2f2"] * len(r)
        return [""] * len(r)
    tbl = tbl.apply(_row, axis=1)
    return (f'<div style="max-height:480px; overflow-y:auto; border:1px solid #e2e8f0; '
            f'border-radius:6px;">{tbl.to_html()}</div>')

# ── widget assembly ────────────────────────────────────────────────────────
if not HAVE_WIDGETS:
    display(HTML('<div style="font:italic 12px/1.4 system-ui; color:#64748b; margin:10px 0;">'
                 "ipywidgets unavailable — this panel needs widgets; in Colab they work by default.</div>"))
else:
    def _img(png_bytes):
        return W.Image(value=png_bytes, format="png",
                       layout=W.Layout(max_width="920px"))

    def _h(text):
        return W.HTML(f'<div style="font:600 13px/1.4 system-ui; color:#0f172a; margin:16px 0 4px;">{text}</div>')

    def _status(t):
        return W.HTML(f'<div style="font:400 11px/1.5 system-ui; color:#64748b; margin:12px 0 0;">'
                      f'FLEET_MONTHLY_FIXED_USD = <b>${t["total_monthly_cost"]:,.0f}</b> — '
                      f'exposed as a global for the downstream break-even overlay.</div>')

    TAB_TITLES = ["Cost by vendor", "Occupancy", "P&L", "Contract expiry", "Detail table"]

    def _build_children(f, t):
        kpi = W.HTML(_kpi_html(t))
        tab = W.Tab(layout=W.Layout(width="960px"))
        tab.children = [
            _img(_fig_cost_bytes(f)),
            _img(_fig_occupancy_bytes(f)),
            _img(_fig_pnl_bytes(t)),
            W.HTML(_expiry_table_html(f)),
            W.HTML(_table_html(f)),
        ]
        for i, name in enumerate(TAB_TITLES):
            tab.set_title(i, name)
        return (kpi, tab, _status(t))

    sliders = {}
    slider_rows = []
    for res in _INTERNAL_RESOURCES:
        row = _base[_base["Resource"] == res].iloc[0]
        init = _default_occ(res)
        init = int(init) if init is not None else int(row["GPU Count"])
        w = W.IntSlider(value=init, min=0, max=int(row["GPU Count"]), step=1,
                        description=res[:44],
                        continuous_update=False,          # fire once on release — no render storm
                        style={"description_width": "300px"},
                        layout=W.Layout(width="640px"))
        sliders[res] = w
        slider_rows.append(w)

    # revenue inputs — one box per customer found in the CSV + a generic "Other"
    _CUSTOMERS = sorted({c for c in _base["Customer"].unique() if c and c != "nan"})
    rev_inputs = {}
    rev_rows = []
    for cust in _CUSTOMERS + ["Other"]:
        cust_rev = float(FLEET_CUSTOMER_MONTHLY_REVENUE.get(cust, 0) or 0)
        w = W.FloatText(value=cust_rev, description=f"{cust} revenue $/mo",
                        continuous_update=False,
                        style={"description_width": "200px"},
                        layout=W.Layout(width="400px"))
        rev_inputs[cust] = w
        rev_rows.append(w)

    result = W.VBox()

    def _rerender(*_):
        rev = {c: w.value for c, w in rev_inputs.items() if w.value}
        f = _compute({r: w.value for r, w in sliders.items()}, rev_overrides=rev)
        t = _totals(f)
        global FLEET_MONTHLY_FIXED_USD
        FLEET_MONTHLY_FIXED_USD = t["total_monthly_cost"]
        result.children = _build_children(f, t)

    for w in list(sliders.values()) + list(rev_inputs.values()):
        w.observe(_rerender, names="value")

    display(W.VBox([
        W.HTML('<div style="font:600 13px/1.3 system-ui; color:#0f172a; margin:4px 0 2px;">'
               "⚙︎ Fleet inputs — drag an occupancy slider or type a customer's monthly revenue; the panel below re-renders in place</div>"),
        W.HTML('<div style="font:600 11px/1.3 system-ui; color:#475569; margin:6px 0 0;">Internal occupancy (GPUs in use)</div>'),
        W.VBox(slider_rows),
        W.HTML('<div style="font:600 11px/1.3 system-ui; color:#475569; margin:10px 0 0;">Customer revenue ($ / month) — renting GPUs out or selling a service both count</div>'),
        W.VBox(rev_rows),
        result,
    ]))
    _rerender()


## 1 · Live calculator — pick workload, GPU table updates inline

Picks a workload (model × ISL × OSL × KV hit rate × per-token prices × GPU class × fleet size × time window) and prints per-offer economics: revenue/h, profit/h, margin %, breakeven utilisation, and the fleet × window totals.


### How the calculator computes profit

**Inputs you set** (via the widget knobs):

| knob | type | meaning |
|---|---|---|
| `model` | dropdown | one of the 5 supported LLMs (see price/anchor provenance below) |
| `ISL` | dropdown | input sequence length in tokens (only 13k / 25k have measured KV levels; the others clip) |
| `OSL` | slider (tokens) | output sequence length in tokens, 128–4096 |
| `KV hit rate %` | text | fraction of input tokens that hit the prefix cache |
| `pin / pca / pou` | text $/1M | fresh-input, cached-input, output prices per million tokens |
| `hardware` | dropdown | GPU class (or "all scaled" overlays the per-GPU table) |
| `GPU utilisation %` | slider 10–100 | share of each GPU-hour that actually sells tokens; revenue scales by u, cost stays fixed. `be_lower` therefore stays the answer to "how busy must I be" — unchanged by this slider |
| `price source` | toggle | mine (your Notion-tracker quotes) vs public market scan vs both |
| `GPUs (nodes)` | text | how many rentals of this option you sell at once |
| `time window (days)` | text | per-offer totals multiplied by `days × 24` hours |

_KV-gain strength slider removed 2026-10-28_: the per-model prior in `KV_GAIN_MODEL_PRIOR` now applies at 100 % always. If you need the sensitivity sweep again, it lives in git history.

**Master formula** (per-GPU, per-hour):

```
revenue_full/h = [in_tps × (1 − kv) × pin + in_tps × kv × pca + out_tps × pou] × 3600 / 1e6 × hw_scale
                      fresh-input channel       cached-input channel   output channel

profit/h = revenue_full/h × utilisation − listed_rate
```

**`be_lower` (breakeven utilisation, lower bound)** is always a ratio between `0` (instant break-even) and `1` (must be 100 % busy to break even). **`be_upper` uses a flat 1.6× multiplier on `be_lower`, with no clipping** — read it as "the utilisation you'd need if your realistic busy ceiling is ≈ 62.5 % of nameplate (1/1.6)". The 1.6× is deliberately conservative: real batching-derate depends on model + concurrency shape and we have no measured surface (the workbook's original ≤5 %–22 % partial-load curve is an idea, not a measurement). Values above 1.0 are legal and informative: 1.60 means "needs 160 % busy ⇒ impossible by 1.6×", 3.20 means "off by 3.2×". An earlier version clipped at 1.0; that collapsed every marginal offer (`be_lower ≥ 0.625`) to the same 1.0, displayed `be_upper < be_lower` on losing rows, and drew negative segments in the §2 stacked chart — so the clip was removed.

#### Where the numbers come from

`margin_pct` is rendered as percent everywhere. `be_lower / be_upper` are unit-less ratios, formatted to 2 decimals, in every table.

| `be_lower` | ratio | breakeven utilisation, linear bound: how busy the GPU must be (0–1) for the offer to break even. 1.0 = must be 100% busy. Below 1.0 = leaves headroom; above = lose money even at full load. |
| `be_upper` | ratio | breakeven utilisation, upper bound: `be_lower × 1.6` (no clip). `be_upper > 1.0` reads "cannot break even under derated load, off by this factor" — e.g. 1.05 ⇒ barely impossible, 3.20 ⇒ 3.2× oversubscribed. The 1.6× penalty is a deliberate flat conservatism — a measured batching-derate curve does not yet exist. Same units as `be_lower`. |


In [ ]:
try:
    import ipywidgets as W
    from IPython.display import display, clear_output, HTML
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False

if not HAVE_WIDGETS or grid_full is None:
    print("ipywidgets or grid_full unavailable — run cell 0+2 first.")
else:
    # Same reload guard as §2: keep this cell working even if grid_full came
    # from an older kernel session.
    if {"be_lower", "be_upper"} - set(grid_full.columns):
        _grid_path = _local("grid_full_v2.csv")
        if _grid_path is None:
            raise FileNotFoundError("grid_full_v2.csv — schema reload needed but file not found anywhere (cwd / ./data / ./grid / notebook dir / Drive)")
        grid_full = pd.read_csv(_grid_path)

    # ── input widgets ─────────────────────────────────────────────────────────
    _MODEL_OPTS = [("GLM-5.3-Flash (measured)", "glm-5.3-flash"),
                   ("Kimi-K3 (public anchor)",  "kimi-k3"),
                   ("DeepSeek-V4.1-Flash",      "deepseek-v4.1-flash"),
                   ("Tencent Hy4-preview",      "hy4-preview"),
                   ("Xiaomi MiMo-V2.6-Flash",   "mimo-v2.6-flash")]
    _GPU_OPTS   = [("all GPUs (scaled)", "__all__")] + [(g.upper(), g) for g in
                ["gb300","b300","mi355x","b200","gb200","h200","mi325x","h100"]]
    _PRICE_OPTS = [("my offers","my_quotes"),("public market","public_market"),("both","both")]
    _ISL_OPTS   = [("ISL 13k", 13000), ("ISL 25k", 25000)]

    c5_w_model   = W.Dropdown(options=_MODEL_OPTS,   value="glm-5.3-flash",  description="model",
                           style={'description_width':'130px'}, layout=W.Layout(width='280px'))
    c5_w_isl     = W.Dropdown(options=_ISL_OPTS,     value=25000,            description="my workload",
                           style={'description_width':'130px'}, layout=W.Layout(width='180px'))
    c5_w_osl     = W.IntSlider(value=512, min=128, max=4096, step=1,
                            description="OSL tokens", continuous_update=False,
                            style={'description_width':'130px'}, layout=W.Layout(width='340px'))
    c5_w_kv      = W.FloatText(value=60.0, description="KV hit rate %",
                             style={'description_width':'130px'}, layout=W.Layout(width='200px'))
    c5_w_pin     = W.FloatText(value=0.15, description="fresh-in $/1M",
                             style={'description_width':'130px'}, layout=W.Layout(width='200px'))
    c5_w_pca     = W.FloatText(value=0.03, description="cached-in $/1M",
                             style={'description_width':'130px'}, layout=W.Layout(width='200px'))
    c5_w_pou     = W.FloatText(value=0.50, description="output $/1M",
                             style={'description_width':'130px'}, layout=W.Layout(width='200px'))
    c5_w_gpu     = W.Dropdown(options=_GPU_OPTS,     value="__all__",        description="hardware",
                           style={'description_width':'130px'}, layout=W.Layout(width='300px'))
    c5_w_price   = W.ToggleButtons(options=_PRICE_OPTS, value="both",
                                description="price source")
    c5_w_fleet   = W.IntText(value=16, description="GPUs (nodes)",
                           style={'description_width':'130px'}, layout=W.Layout(width='200px'))
    c5_w_horizon = W.IntText(value=7,  description="time window (days)",
                           style={'description_width':'130px'}, layout=W.Layout(width='200px'))
    c5_w_util    = W.IntSlider(value=100, min=10, max=100, step=5,
                            description="GPU utilisation %", continuous_update=False,
                            style={'description_width':'130px'}, layout=W.Layout(width='280px'))

    c5_output = W.VBox()

    # ── price/anchor dicts (same values as markdown § How the calculator computes profit) ──
    PRICES = {"glm-5.3-flash": dict(pin=.15, pca=.03, pou=.50),
              "kimi-k3":       dict(pin=3.,  pca=.30, pou=15.),
              "deepseek-v4.1-flash": dict(pin=.15, pca=.003, pou=.60),
              "hy4-preview":   dict(pin=.80, pca=.038, pou=2.48),
              "mimo-v2.6-flash": dict(pin=.14, pca=.003, pou=.28)}
    HW_SCALE = {"glm-5.3-flash": {"b300":1.,"gb300":1.12,"b200":.454,"gb200":.187,
                                  "h200":.049,"mi355x":.493,"mi325x":.026,"h100":.041},
                "kimi-k3":       {"b300":1.,"gb300":5.877,"b200":.988,"gb200":1.068,
                                  "h200":.012,"mi355x":1.213,"mi325x":.669,"h100":.063},
                "deepseek-v4.1-flash": {"b300":1.,"gb300":5.936,"b200":.883,"gb200":.831,
                                         "h200":.159,"mi355x":.426,"mi325x":.090,"h100":.041},
                "hy4-preview":   {"b300":1.,"gb300":1.132,"b200":.801,"gb200":.801,
                                  "h200":.215,"mi355x":.878,"mi325x":0.,"h100":0.},
                "mimo-v2.6-flash":{"b300":1.,"gb300":1.,"b200":.6,"gb200":.55,
                                   "h200":.1,"mi355x":.5,"mi325x":.08,"h100":.06}}
    # Provenance per non-unity multiplier above (b300=1.0 is the anchor row).
    #   bench    = first-party measurement (own B300 lab / InferenceX csv),
    #   ratchet  = derived from a related-SKU ratio on a different model,
    #   heuristic= tuned off MEM/SM spec when no bench exists.
    HW_SCALE_SOURCE = {
        "glm-5.3-flash":      {"gb300":"bench:infx_glm53_2026-10-20","b200":"bench:infx_glm53_2026-10-20",
                                "gb200":"bench:infx_glm53_2026-10-20","h200":"bench:infx_glm53_2026-10-20",
                                "mi355x":"bench:infx_glm53_2026-10-20","mi325x":"bench:infx_glm53_2026-10-20",
                                "h100":"bench:infx_glm53_2026-10-20"},
        "kimi-k3":            {"gb300":"bench:infx_kimi_k3_2026-10-27","b200":"bench:infx_kimi_k3_2026-10-26",
                                "gb200":"bench:infx_kimi_k3_2026-10-21","h200":"bench:infx_kimi_k3_2026-08-07",
                                "mi355x":"bench:infx_kimi_k3_2026-10-27","mi325x":"holdover:infx_kimi_k3_2026-10-20",
                                "h100":"holdover:infx_kimi_k3_2026-10-20"},
        "deepseek-v4.1-flash":{"gb300":"bench:infx_dsv41_flash_2026-10-20","b200":"bench:infx_dsv41_flash_2026-11-05",
                                "gb200":"bench:infx_dsv41_flash_2026-11-09","h200":"bench:infx_dsv41_flash_2026-11-09",
                                "mi355x":"bench:infx_dsv41_flash_2026-11-09","mi325x":"bench:infx_dsv41_flash_2026-11-06",
                                "h100":"bench:infx_dsv41_flash_2026-11-06"},
        "hy4-preview":        {"gb300":"bench:infx_musespark_2026-10-27","b200":"bench:infx_musespark_2026-11-06",
                                "gb200":"bench:infx_musespark_2026-11-06","h200":"bench:infx_musespark_2026-10-19",
                                "mi355x":"bench:infx_musespark_2026-11-08","mi325x":"unsupported (musespark n/a)","h100":"unsupported (musespark n/a)"},
        "mimo-v2.6-flash":    {"gb300":"ratchet:hy4-shape × mi355x/hy4-b200","b200":"heuristic:b200=gb200",
                                "gb200":"heuristic:h100-shape on b300","h200":"heuristic:hy4-h200",
                                "mi355x":"heuristic:hy4-mi355x","mi325x":"heuristic:mi325x=mi355x-shaped","h100":"heuristic:h100=mi325x-shaped"},
    }
    GATES = {25000: {60:{128:(21957.5,112.4),256:(19472.5,199.4),512:(21998.,450.5),1024:(5707.5,233.7)},
                      90:{128:(52767.5,270.2),256:(34350.,351.7),512:(10750.,220.1),1024:(2905.,119.)},
                      92:{128:(46405.,237.6),256:(35470.,363.2),512:(18957.5,388.3),1024:(4917.5,201.5)}},
             13000: {60:{128:(20718.1,204.),256:(18848.7,371.2),512:(10095.8,397.6),1024:(3006.9,236.8)},
                      89:{128:(33507.5,329.9),256:(31311.8,616.6),512:(18250.7,718.8),1024:(3181.1,250.6)}}}
    ANCHOR = {"kimi-k3":            dict(in_tps=12479., out_tps=87.6),
              "deepseek-v4.1-flash": dict(in_tps=441986., out_tps=3170.5),
              "hy4-preview":        dict(in_tps=54484.,out_tps=379.4),
              "mimo-v2.6-flash":    dict(in_tps=33629.,out_tps=250.)}
    # refreshed 2026-11-09 from data/infx_all_models_anchors.csv

    KV_GAIN_PTS = {(60,128):1.00,(60,256):1.00,(60,512):1.00,(60,1024):1.00,
                   (90,128):2.40,(90,256):1.76,(90,512):0.87,(90,1024):0.51}

    # ── Per-model KV-gain prior ──────────────────────────────────────────────
    # KV_GAIN_PTS comes from the user's GLM-5.3 measured sweep, so
    # kv_gain_at(...) returns 'GLM-shape gain' for any (kv, osl). Different
    # public models have different attention (MLA / GQA+CLA / SWA+GA / Mamba),
    # so we scale GLM's shape by a per-model prior derived from active-params
    # sparsity + KV-scheme + published architecture papers:
    #
    #   deepseek-v4.1-flash : 1.00  DeepSeek-V3 Day-6 report (H800: prefill
    #                                73.7k tok/s per node vs decode 14.8k, 5.0x
    #                                ratio, 56.3% production hit rate).
    #                                671B/37B (5.5% active) matches GLM-5.3's
    #                                744B/40B (5.4%), same MLA family. Anchor.
    #   kimi-k3             : 1.30  Mooncake paper (FAST'25): hit-rate vs
    #                                throughput +20% at 0%, +40% at 80%,
    #                                +525% simulated ceiling. K2 = 1T/32B
    #                                (3.2% active) is sparser than GLM → cache
    #                                hits save more prefill share. K3 has no
    #                                first-party data.
    #   hy4-preview         : 1.00  No public checkpoint matches '770B/49B'.
    #                                Closest public = Hunyuan-Large (389B/52B,
    #                                13.4% active) using GQA+CLA (95% KV
    #                                reduction absolute ≈ MLA sparsity-wise).
    #   mimo-v2.6-flash     : 0.50  MiMo-V2-Flash arXiv 2601.02780: 309B/15B
    #                                (4.9% ≈ GLM sparsity), BUT 39 SWA + 9 GA
    #                                layers (NOT MLA). SWA windows (W=128)
    #                                cap long-context prefill work to save →
    #                                cache-hit gain ≈ half of GLM's.
    #
    # The slider KV-gain strength % is a sensitivity knob on top of this:
    # effective_gain = kv_gain_at(kv, osl, strength * prior).
    KV_GAIN_MODEL_PRIOR = {
        "kimi-k3":              1.30,
        "deepseek-v4.1-flash":  1.00,
        "hy4-preview":          1.00,
        "mimo-v2.6-flash":      0.50,
    }

    def _kv_gain_at(kv, osl, strength=1.0):
        kvs  = sorted({p[0] for p in KV_GAIN_PTS})
        osls = sorted({p[1] for p in KV_GAIN_PTS})
        kv_c  = max(kvs[0],  min(kv,  kvs[-1]));  osl_c = max(osls[0], min(osl, osls[-1]))
        kv_lo  = max(k for k in kvs if k <= kv_c);  kv_hi  = min(k for k in kvs if k >= kv_c)
        osl_lo = max(o for o in osls if o <= osl_c); osl_hi = min(o for o in osls if o >= osl_c)
        f11 = KV_GAIN_PTS[(kv_lo, osl_lo)]; f12 = KV_GAIN_PTS[(kv_lo, osl_hi)]
        f21 = KV_GAIN_PTS[(kv_hi, osl_lo)]; f22 = KV_GAIN_PTS[(kv_hi, osl_hi)]
        tx = 0 if kv_hi==kv_lo  else (kv_c - kv_lo) / (kv_hi - kv_lo)
        ty = 0 if osl_hi==osl_lo else (osl_c - osl_lo) / (osl_hi - osl_lo)
        raw = f11*(1-tx)*(1-ty) + f21*tx*(1-ty) + f12*(1-tx)*ty + f22*tx*ty
        return 1.0 + (raw - 1.0) * strength

    def _last_sync():
        return _synced.get('model')
    def _mark_synced(m):
        _synced['model'] = m

    _synced = {'model': None}
    _c5_rendering = {'active': False}

    def c5_run(_=None):
        # Render via c5_output.outputs directly so a single logical event can't
        # end up queued twice when our own writes retrig observers.
        # (ipywidgets' Output context manager can double-draw if we mutate the
        # observed sliders inside the handler.)
        if _c5_rendering.get("active"):
            return
        _c5_rendering["active"] = True
        try:
            return _c5_render_body(_)
        finally:
            _c5_rendering["active"] = False

    def _c5_render_body(_=None):
        # Render into a VBox we own; replace its .children atomically so widget events
        # that queue up while we're rendering can't stack duplicates into the cell output.
        _new_children = []
        def _push(w):
            _new_children.append(w)
        _push_text = lambda s: _push(W.HTML(
            f'<pre style="font:12px/1.4 ui-monospace,monospace; color:#0f172a; margin:2px 0; white-space:pre-wrap;">{s}</pre>'))
        if True:
            MODEL   = c5_w_model.value
            if MODEL != _last_sync():
                _mark_synced(MODEL)            # mark FIRST so observer re-entry exits immediately
                pq = PRICES[MODEL]
                c5_w_pin.value = pq['pin']
                c5_w_pca.value = pq['pca']
                c5_w_pou.value = pq['pou']
            ISL     = c5_w_isl.value
            OSL     = int(c5_w_osl.value)
            KV_HIT  = float(c5_w_kv.value)
            N_GPUS  = int(c5_w_fleet.value)
            DAYS    = float(c5_w_horizon.value)
            UTIL    = float(c5_w_util.value)/100.   # share of each GPU-hour actually selling tokens
            P       = dict(pin=c5_w_pin.value, pca=c5_w_pca.value, pou=c5_w_pou.value)
            PRICE_OPT = c5_w_price.value

            hw = HW_SCALE[MODEL]
            scale = 1.0 if c5_w_gpu.value == "__all__" else hw.get(c5_w_gpu.value, 0.)

            # strength slider removed (2026-10-28): sensitivity to the unknown
            # KV-gain prior was rarely exercised and cluttered the UI; locked at
            # 100 % (prior-only). KV_GAIN_MODEL_PRIOR still applies per model.
            strength = 1.0

            if MODEL == "glm-5.3-flash":
                isl_gates = GATES.get(ISL)
                if isl_gates is None: isl_gates = GATES[25000]
                kv_tbl = isl_gates
                kvs = sorted(kv_tbl.keys())
                kv_use = kvs[-1] if KV_HIT >= kvs[-1] else (kvs[0] if KV_HIT <= kvs[0] else max(k for k in kvs if k <= KV_HIT))
                osl_tbl = kv_tbl[kv_use]
                osls = sorted(osl_tbl.keys())
                osl_use = osls[-1] if OSL >= osls[-1] else (osls[0] if OSL <= osls[0] else OSL)
                o_anchor = max(o for o in osls if o <= osl_use)
                in_t, out_t = osl_tbl[o_anchor]
                gain = 1.0
                note = "measured"
            else:
                a = ANCHOR[MODEL]
                in_t, out_t = a["in_tps"], a["out_tps"]
                prior = KV_GAIN_MODEL_PRIOR[MODEL]
                gain  = _kv_gain_at(KV_HIT, OSL, strength=strength * prior)
                in_t  *= gain
                out_t *= gain
                note = f"public anchor × {gain:.2f} KV-gain (strength={strength:.0%} × model-prior={prior:.2f})"

            # per-GPU scaling happens per row below — keep the B300 anchor here
            # pure so the header line stays readable.
            hw_label = ("all GPUs (each row scaled by its own hw_scale vs B300)"
                        if c5_w_gpu.value == "__all__"
                        else f"only {c5_w_gpu.value.upper()} · t/s x{hw.get(c5_w_gpu.value, 1.0)}")

            k = KV_HIT/100.
            def _rev_h(in_v, out_v):
                return (in_v*(1-k)*P['pin'] + in_v*k*P['pca'] + out_v*P['pou']) * 3600/1e6

            in_anchor, out_anchor = in_t, out_t   # B300 anchors (pre per-GPU scale)
            rev_h_b300            = _rev_h(in_anchor, out_anchor)

            if c5_w_gpu.value != "__all__":
                in_t  = in_anchor  * hw.get(c5_w_gpu.value, 1.0)
                out_t = out_anchor * hw.get(c5_w_gpu.value, 1.0)
                rev_h = _rev_h(in_t, out_t)
            else:
                in_t, out_t = in_anchor, out_anchor

            _push_text(f"WORKLOAD  {MODEL}  isl={ISL:,}  osl={OSL}  KV={KV_HIT}%  ·  KV-gain {gain:.2f} (strength {strength:.0%})   [{note}]")
            _push_text(f"  B300 anchor: in {in_anchor:,.0f} t/s + out {out_anchor:,.1f} t/s = {in_anchor+out_anchor:,.0f} mixed t/s  (revenue ${rev_h_b300:,.3f}/GPU·h on B300)")
            _push_text(f"  prices: fresh-in ${P['pin']}/1M · cached-in ${P['pca']}/1M · out ${P['pou']}/1M")
            _official = PRICES[MODEL]
            _at_official = (abs(P['pin']-_official['pin'])<1e-9 and
                            abs(P['pca']-_official['pca'])<1e-9 and
                            abs(P['pou']-_official['pou'])<1e-9)
            _official_str = (f"fresh ${_official['pin']:g} · cached ${_official['pca']:g} · out ${_official['pou']:g}")
            _tag = 'official' if _at_official else 'OVERRIDDEN'
            _push_text(f"    · {MODEL} rate card: {_official_str} $/1M  [{_tag}]")
            _push_text(f"  hardware: {hw_label}")
            _push_text(f"  fleet: {N_GPUS} GPU  ×  window: {DAYS} days ({int(DAYS*24)} h)  ×  utilisation {UTIL:.0%}  →  $ totals below are for that size × window × util")

            d = grid_full[(grid_full.model==MODEL)
                          & (grid_full.mix=="agentic")
                          & (grid_full.throughput_case=="best")].copy()
            if PRICE_OPT != "both": d = d[d.price_src == PRICE_OPT]
            if c5_w_gpu.value != "__all__": d = d[d.gpu_class == c5_w_gpu.value]
            d = d.drop_duplicates(subset=["option_id","gpu_class","price_src"])

            if not len(d):
                _push_text("no rows for this combination")
                c5_output.children = tuple(_new_children)
                return

            if c5_w_gpu.value == "__all__":
                hw_row = d.gpu_class.map(lambda g: hw.get(g, 1.0))
                d["revenue_ph"]   = (in_anchor  * hw_row * (1-k) * P['pin']
                                   + in_anchor  * hw_row * k     * P['pca']
                                   + out_anchor * hw_row          * P['pou']) * 3600/1e6
            else:
                d["revenue_ph"]   = rev_h
            # revenue_ph stays the FULL-load number (be_lower = cost/revenue is
            # exactly "utilisation needed to break even"). Profit applies UTIL on top.
            d["profit_ph"]      = (d.revenue_ph * UTIL) - d.eff_cost_ph
            d = d.dropna(subset=["revenue_ph"])   # GPU this model can't run → hw_scale missing → drop
            d["margin_pct"]     = (d.profit_ph / (d.revenue_ph * UTIL) * 100).round(1)
            d["be_lower"]       = (d.eff_cost_ph / d.revenue_ph).round(3)
            hours = DAYS*24
            d["revenue_total"] = round(d.revenue_ph * UTIL * N_GPUS * hours, 0)
            d["cost_total"]    = round(d.eff_cost_ph * N_GPUS * hours, 0)
            d["profit_total"]  = round(d.profit_ph  * N_GPUS * hours, 0)
            if c5_w_gpu.value == "__all__":
                d["hw_scale"]      = d.gpu_class.map(lambda g: hw.get(g, 0.0))
                d["fresh_in_tps"]  = (in_anchor  * d["hw_scale"] * (1-k)).round(0)
                d["cached_in_tps"] = (in_anchor  * d["hw_scale"] *  k   ).round(0)
                d["out_tps"]       = (out_anchor * d["hw_scale"]).round(1)
            else:
                d["hw_scale"]      = hw.get(c5_w_gpu.value, 1.0)
                d["fresh_in_tps"]  = round(in_t*(1-k), 0)
                d["cached_in_tps"] = round(in_t*k,    0)
                d["out_tps"]       = round(out_t,       1)

            # Render EVERY row, but inside a scrollable div (~15 rows tall).
            # Users can scroll the widget output to see beyond the first 15.
            view = (d[["option_id","profit_ph","margin_pct","revenue_ph",
                       "profit_total","cost_total","revenue_total",
                       "be_lower","listed_rate","gpu_class","hw_scale","rental_term",
                       "fresh_in_tps","cached_in_tps","out_tps","price_src"]]
                    .sort_values("profit_total", ascending=False))
            _push_text(f"  (showing all {len(view):,} rows in a scrollable view — top-ranked first)")
            _styler5 = (view.style
                    .format({"listed_rate":"${:,.2f}",
                             "revenue_ph":"${:,.3f}","profit_ph":"${:,.3f}",
                             "revenue_total":"${:,.0f}","cost_total":"${:,.0f}","profit_total":"${:,.0f}",
                             "be_lower":"{:.1%}", "hw_scale":"{:.3f}",
                             "fresh_in_tps":"{:,.0f}","cached_in_tps":"{:,.0f}","out_tps":"{:,.1f}",
                             "margin_pct":"{:.1f}%"},
                            na_rep="—")
                    .set_properties(**{"text-align":"right"})
                    .set_properties(subset=["option_id","price_src"],**{"text-align":"left"})
                    .set_properties(subset=["option_id"],**{"min-width":"280px","max-width":"340px",
                                                             "white-space":"normal","word-break":"break-word"}))
            _push(W.HTML(f'<div style="max-height:400px; overflow-y:auto; border:1px solid #ccc; border-radius:4px;">{_styler5.to_html()}</div>'))

        c5_output.children = tuple(_new_children)

    for _w in (c5_w_model, c5_w_isl, c5_w_osl, c5_w_kv, c5_w_pin, c5_w_pca, c5_w_pou,
                c5_w_gpu, c5_w_price, c5_w_fleet, c5_w_horizon, c5_w_util):
        _w.observe(c5_run, "value")

    display(W.VBox([
        W.HBox([c5_w_model, c5_w_isl, c5_w_osl, c5_w_kv]),
        W.HBox([c5_w_pin, c5_w_pca, c5_w_pou]),
        W.HBox([c5_w_gpu, c5_w_util]),            # capacity knobs: hardware + utilisation
        W.HBox([c5_w_price, c5_w_fleet, c5_w_horizon]),
        c5_output
    ]))
    c5_run()

## 2 · Recommendation panel — pick model / GPU / price source / workload, see the best-ranked offer

Plain text — set the four knobs below and re-run:

* `MODEL` — single model name · `__all__` (5-model rollup) · `__compare__` (mine − public)
* `PRICE_SRC` — `my_quotes` · `public_market` · `__all__` (both)
* `GPU_CLASS` — single GPU family or `__all__`
* `WORKLOAD` — `(mix, ISL, OSL, KV_hit_pct)` as a 4-tuple
* `UTIL` — GPU utilisation % (10–100). Scales revenue; `be_lower`/`be_upper` stay utilisation-independent ("how busy must I be")

The widget keys on the same `grid_full_v2.csv` that the live calculator reads, so numbers reconcile 1:1.

#### Suggested token prices — what each column means

When you pick **a single model** (not "All"/"Compare"), the panel additionally
shows, for every GPU family, what you should charge per 1M tokens to host this
model yourself. All pricing columns are USD per 1M tokens under the current
agentic mix (fresh-in 34%, cached-in 65%, output 1%):

* `gpu` — GPU family being priced.
* `offer` — which rental row from grid gives the lowest cost on this GPU.
* `cost_h` — effective hourly cost of that offer ($/GPU·h), already amortised from upfront fees.
* `tps` — blended tokens/sec on this GPU for the current workload (from §1's anchor × hw_scale).
* `be_price` — **breakeven price**: the minimum $/1M tokens at which rental cost == revenue.
  Selling below this loses money. Selling at this price = zero margin.
* `price@30%gm` / `price@50%gm` / `price@70%gm` — what $/1M you must charge to keep
  30% / 50% / 70% gross margin. Each is `be_price / (1 − margin)`. Higher margin tier =
  higher price, obviously — the question is whether the public market will pay it.
* `headroom@70%` — `(public blended price) / (price@70%gm) − 1`. **Positive**
  (green) = the public market is already paying above your 70%-GM ask, so you have room
  to undercut while still taking 70% GM. **Negative** (red) = the public market already
  pays less than you need, so to compete you'd have to drop below 70% GM or pick a
  cheaper GPU offer.

The line above the table repeats the model's **public list price** (fresh / cached / output $/1M)
from OpenRouter or the model's official rate card, blended with the agentic mix weights
— that's the market comparison anchor for `headroom@70%`.

#### Where all numbers come from

* `tps` — from §1's `ANCHOR` dict for public-anchor models, or the measured GATES table for GLM-5.3.
* HW scaling — same `HW_SCALE` dict §1 uses (per-model, per-GPU ratio vs B300).
* `cost_h` — your Notion quotes (`price_src=my_quotes`) and the public GPU price scan
  (`price_src=public_market`), the same grid `eff_cost_ph` column the rest of the notebook uses.
* Public list price for each model — `PRICES` dict in cell 5, sourced from
  OpenRouter listings and provider rate cards (see cell 5 comments).


In [36]:
import io
# ── Cell-level constants used inside c7_run (defined here so any function in this cell can read them) ──
# Per-workload mix fractions: fresh/cached/out share of total tokens.
# Source: back-of-envelope — agentic runs are dominated by long-context cache
# hits (Hy4-preview agentic case); chatisy is dominated by fresh prompts and
# many short generated spans. **Do not collapse these — the recommendation
# panel's per-row revenue is meaningless without mix-correct fractions.**
_MIX_FRACTIONS  = {
    'agentic': dict(fresh=0.34, cached=0.65, out=0.01),
    'chatisy': dict(fresh=0.55, cached=0.20, out=0.25),
}
_CLASS_WEIGHT  = dict(fresh=1.0, cached=0.20, out=1.0)   # heuristic — cached tokens are ~5× cheaper on the GPU
_SUG_MARGIN_TIERS = [('30', 0.30), ('50', 0.50), ('70', 0.70)]

try:
    import ipywidgets as W
    from IPython.display import display, clear_output, HTML
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False

if not HAVE_WIDGETS or grid_full is None:
    print("ipywidgets or grid_full unavailable — run cell 0+2 first.")
else:
    # If the in-memory grid_full came from an older kernel session it may still
    # carry the legacy per-call column names. Reload from disk so this cell
    # always sees the be_lower/be_upper schema.
    if {"be_lower", "be_upper"} - set(grid_full.columns):
        _grid_path = _local("grid_full_v2.csv")
        if _grid_path is None:
            raise FileNotFoundError("grid_full_v2.csv — schema reload needed but file not found anywhere (cwd / ./data / ./grid / notebook dir / Drive)")
        grid_full = pd.read_csv(_grid_path)

    MODELS_5 = sorted(grid_full.model.unique().tolist())
    PRICE_SRCS = sorted(grid_full.price_src.unique().tolist())
    GPUS_ALL = sorted(grid_full.gpu_class.unique().tolist())
    COMPARE = "__compare__"

    c7_w_model   = W.Dropdown(options=[("All 5 models (aggregated)","__all__"),
                                    ("Mine − Public per GPU · your margin edge", COMPARE)] +
                                   [(m, m) for m in MODELS_5],
                           value="__all__", description="model",
                           style={'description_width':'130px'}, layout=W.Layout(width='340px'))
    c7_w_price   = W.Dropdown(options=[("all", "__all__")] + [(v, v) for v in PRICE_SRCS],
                           value="__all__", description="price source",
                           style={'description_width':'130px'}, layout=W.Layout(width='200px'))
    c7_w_gpu     = W.Dropdown(options=[("all", "__all__")] + [(g2.upper(), g2) for g2 in GPUS_ALL],
                           value="__all__", description="GPU class",
                           style={'description_width':'130px'}, layout=W.Layout(width='200px'))
    c7_w_work    = W.Dropdown(options=[
        ("agentic · ISL 25k · OSL 512 · KV 60%",  ("agentic",  25000, 512, 60)),
        ("agentic · ISL 25k · OSL 128 · KV 90%",  ("agentic",  25000, 128, 90)),
        ("agentic · ISL 25k · OSL 1024 · KV 60%", ("agentic",  25000, 1024, 60)),
        ("agentic · ISL 13k · OSL 512 · KV 60%",  ("agentic",  13000, 512, 60)),
        ("agentic · ISL 13k · OSL 256 · KV 89%",  ("agentic",  13000, 256, 89)),
        ("chat · ISL 4k · OSL 128 · KV 40%",      ("chatisy",   4000, 128, 40)),
        ("chat · ISL 1k · OSL 128 · KV 60%",      ("chatisy",   1000, 128, 60)),
    ], value=("agentic", 25000, 512, 60), description="workload",
        style={'description_width':'130px'}, layout=W.Layout(width='360px'))
    c7_w_util    = W.IntSlider(value=100, min=10, max=100, step=5,
                            description="GPU utilisation %", continuous_update=False,
                            style={'description_width':'130px'}, layout=W.Layout(width='280px'))

    _c7_rendering = {'active': False}
    c7_output = W.VBox()

    def c7_run(_=None):
        if _c7_rendering.get("active"):
            return
        _c7_rendering["active"] = True
        try:
            return _c7_render_body(_)
        finally:
            _c7_rendering["active"] = False

    def _c7_render_body(_=None):
        # Render into a VBox we own; replace its .children atomically so widget events
        # that queue up while we're rendering can't stack duplicates into the cell output.
        _new_children = []
        def _push(w):
            _new_children.append(w)
        _push_text = lambda s: _push(W.HTML(
            f'<pre style="font:12px/1.4 ui-monospace,monospace; color:#0f172a; margin:2px 0; white-space:pre-wrap;">{s}</pre>'))
        if True:
            # Force-read the widget values each call. This guarantees the
            # pricing block below sees fresh slider values regardless of
            # how traitlets observers schedule updates.
            mix, isl, osl, kv = c7_w_work.value
            UTIL = float(c7_w_util.value)/100.   # share of each GPU-hour actually selling tokens
            _MIX_FRACTION = _MIX_FRACTIONS[mix]   # per-render binding — picks agentic or chatisy shape

            _HW_SCALE_LOCAL = {
                "glm-5.3-flash":     {"b300":1.,"gb300":1.12,"b200":.454,"gb200":.187,
                                      "h200":.049,"mi355x":.493,"mi325x":.026,"h100":.041},
                "kimi-k3":           {"b300":1.,"gb300":5.877,"b200":.988,"gb200":1.068,
                                      "h200":.012,"mi355x":1.213,"mi325x":.669,"h100":.063},
                "deepseek-v4.1-flash": {"b300":1.,"gb300":5.936,"b200":.883,"gb200":.831,
                                        "h200":.159,"mi355x":.426,"mi325x":.090,"h100":.041},
                "hy4-preview":       {"b300":1.,"gb300":1.132,"b200":.801,"gb200":.801,
                                      "h200":.215,"mi355x":.878,"mi325x":0.,"h100":0.},
                "mimo-v2.6-flash":   {"b300":1.,"gb300":1.,"b200":.6,"gb200":.55,
                                      "h200":.1,"mi355x":.5,"mi325x":.08,"h100":.06}}
            # refreshed 2026-11-09 to match cell-7 HW_SCALE (infx_all_models_anchors.csv)
            _PRIOR_LOCAL = {"kimi-k3":1.30,"deepseek-v4.1-flash":1.00,
                            "hy4-preview":1.00,"mimo-v2.6-flash":0.50}
            _KV_GAIN_PTS_LOCAL = {(60,128):1.,(60,256):1.,(60,512):1.,(60,1024):1.,
                                  (90,128):2.40,(90,256):1.76,(90,512):0.87,(90,1024):0.51}
            def _kv_gain_local(kv, osl, strength=1.0):
                kvs  = sorted({p[0] for p in _KV_GAIN_PTS_LOCAL})
                osls = sorted({p[1] for p in _KV_GAIN_PTS_LOCAL})
                kv_c  = max(kvs[0],  min(kv,  kvs[-1]));  osl_c = max(osls[0], min(osl, osls[-1]))
                kv_lo  = max(k for k in kvs if k <= kv_c);  kv_hi  = min(k for k in kvs if k >= kv_c)
                osl_lo = max(o for o in osls if o <= osl_c); osl_hi = min(o for o in osls if o >= osl_c)
                f11 = _KV_GAIN_PTS_LOCAL[(kv_lo, osl_lo)]; f12 = _KV_GAIN_PTS_LOCAL[(kv_lo, osl_hi)]
                f21 = _KV_GAIN_PTS_LOCAL[(kv_hi, osl_lo)]; f22 = _KV_GAIN_PTS_LOCAL[(kv_hi, osl_hi)]
                tx = 0 if kv_hi==kv_lo  else (kv_c - kv_lo) / (kv_hi - kv_lo)
                ty = 0 if osl_hi==osl_lo else (osl_c - osl_lo) / (osl_hi - osl_lo)
                raw = f11*(1-tx)*(1-ty) + f21*tx*(1-ty) + f12*(1-tx)*ty + f22*tx*ty
                return 1.0 + (raw - 1.0) * strength
            # b300 anchors (in_tps, out_tps) — same as §1 cell 5
            _ANCHOR_LOCAL = {
                "kimi-k3":              dict(in_tps=12479.,  out_tps=87.6),
                "deepseek-v4.1-flash":  dict(in_tps=441986., out_tps=3170.5),
                "hy4-preview":          dict(in_tps=54484.,  out_tps=379.4),
                "mimo-v2.6-flash":      dict(in_tps=33629.,  out_tps=250.),
            }
            # refreshed 2026-11-09 from data/infx_all_models_anchors.csv — mirror of cell-7 ANCHOR
            # GLM gates (measured). Same numbers as cell 5's GATES dict.
            _GATES_LOCAL = {25000: {60:{128:(21957.5,112.4),256:(19472.5,199.4),512:(21998.,450.5),1024:(5707.5,233.7)},
                                   90:{128:(52767.5,270.2),256:(34350.,351.7),512:(10750.,220.1),1024:(2905.,119.)},
                                   92:{128:(46405.,237.6),256:(35470.,363.2),512:(18957.5,388.3),1024:(4917.5,201.5)}},
                            13000: {60:{128:(20718.1,204.),256:(18848.7,371.2),512:(10095.8,397.6),1024:(3006.9,236.8)},
                                   89:{128:(33507.5,329.9),256:(31311.8,616.6),512:(18250.7,718.8),1024:(3181.1,250.6)}}}

            def _b300_anchors_for(model, isl, osl, kv, kv_strength=1.0):
                # Returns (in_tps, out_tps, gain) on B300 for the chosen model
                # and the chosen workload sliders. Mirrors §1 logic.
                if model == "glm-5.3-flash":
                    isl_gates = _GATES_LOCAL.get(isl) or _GATES_LOCAL[25000]
                    kvs = sorted(isl_gates.keys())
                    kv_use = kvs[-1] if kv >= kvs[-1] else (kvs[0] if kv <= kvs[0] else max(k for k in kvs if k <= kv))
                    osl_tbl = isl_gates[kv_use]
                    osls = sorted(osl_tbl.keys())
                    o_anchor = max(o for o in osls if o <= (osl if osl <= osls[-1] else osls[-1]))
                    it, ot = osl_tbl[o_anchor]
                    return it, ot, 1.0
                a = _ANCHOR_LOCAL[model]
                prior = _PRIOR_LOCAL[model]
                gain  = _kv_gain_local(kv, osl, strength=kv_strength * prior)
                return a["in_tps"] * gain, a["out_tps"] * gain, gain



            _push_text(f'[slider read] mix={mix}  isl={isl:,}  osl={osl:,}  kv={kv}%')
            d = grid_full.copy()
            d = d.drop_duplicates(subset=["option_id","gpu_class","price_src","model","mix","throughput_case"], keep="first")
            if c7_w_model.value not in ("__all__", COMPARE): d = d[d.model==c7_w_model.value]
            if c7_w_price.value != "__all__":               d = d[d.price_src == c7_w_price.value]
            if c7_w_gpu.value   != "__all__":               d = d[d.gpu_class == c7_w_gpu.value]
            d = d[d.mix==mix]

            _push_text(f"workload  mix={mix}  ISL={isl:,}  OSL={osl:,}  KV hit={kv}%")
            if not len(d):
                _push_text("no rows for this combination")
                c7_output.children = tuple(_new_children)
                return

            if c7_w_model.value == COMPARE:
                a = (d[d.price_src=='my_quotes']
                     .groupby('gpu_class').profit_ph.max().rename('mine: best profit $/GPU·h'))
                b = (d[d.price_src=='public_market']
                     .groupby('gpu_class').profit_ph.max().rename('public: best profit $/GPU·h'))
                cmp_t = pd.concat([a, b], axis=1).round(2)
                cmp_t['delta (mine − public)'] = (cmp_t.iloc[:,0] - cmp_t.iloc[:,1]).round(2)
                cmp_t = cmp_t.sort_values('delta (mine − public)', ascending=False)
                # print plain-text fallback so the result is visible even if the
                # Styler/widget subtree fails to render
                _styler_cmp = (cmp_t.style
                        .format("{:+.2f}", subset='delta (mine − public)')
                        .format("${:,.2f}", subset=list(cmp_t.columns[:2]))
                        .set_caption("mine − public best-profit per GPU"))
                _push(W.HTML(_styler_cmp.to_html()))
                c7_output.children = tuple(_new_children)
                return

            # Recompute revenue/profit/margin/be_lower per row using the current (isl, osl, kv)
            # sliders — without this, the main table shows static values and won't respond
            # to workload changes.
            work_isl, work_osl, work_kv = isl, osl, kv
            _PRICES_LOCAL = {
                "glm-5.3-flash":       dict(pin=0.15, pca=0.03, pou=0.50),
                "kimi-k3":             dict(pin=3.00, pca=0.30, pou=15.00),
                "deepseek-v4.1-flash": dict(pin=0.15, pca=0.003, pou=0.60),
                "hy4-preview":         dict(pin=0.80, pca=0.038, pou=2.48),
                "mimo-v2.6-flash":     dict(pin=0.14, pca=0.003, pou=0.28),
            }
            _model_anchors = {m: _b300_anchors_for(m, work_isl, work_osl, work_kv)
                              for m in d.model.unique()}
            _inputs, _outputs = [], []
            for _, rr in d.iterrows():
                it, ot, _g = _model_anchors[rr.model]
                hwf = _HW_SCALE_LOCAL[rr.model].get(rr.gpu_class, 0.0)
                _inputs.append(it * hwf)
                _outputs.append(ot * hwf)
            d = d.assign(_in_t=_inputs, _out_t=_outputs)
            _kf, _kc, _ko = _MIX_FRACTION['fresh'], _MIX_FRACTION['cached'], _MIX_FRACTION['out']
            _P_pin = d.model.map(lambda m: _PRICES_LOCAL[m]['pin'])
            _P_pca = d.model.map(lambda m: _PRICES_LOCAL[m]['pca'])
            _P_pou = d.model.map(lambda m: _PRICES_LOCAL[m]['pou'])
            _fsh = d._in_t * _kf / max(_kf + _kc, 1e-9)
            _cch = d._in_t * _kc / max(_kf + _kc, 1e-9)
            d["mixed_tps_per_gpu"] = (d._in_t + d._out_t).round(0)
            # revenue_ph stays FULL-load (be_lower = cost/revenue = breakeven utilisation).
            # profit applies the user-set UTIL on top of that.
            d["revenue_ph"]  = ((_fsh * _P_pin + _cch * _P_pca + d._out_t * _P_pou) * 3600 / 1e6).round(3)
            d["profit_ph"]   = (d.revenue_ph * UTIL - d.eff_cost_ph).round(3)
            d["margin_pct"]  = (d.profit_ph / (d.revenue_ph * UTIL) * 100).round(1)
            d["be_lower"]    = (d.eff_cost_ph / d.revenue_ph).round(3)
            # be_upper = be_lower x 1.6 flat derate ceiling (realistic busy ceiling ~62.5%
            # of nameplate). NO clip at 1.0: clipping collapsed every be_lower >= 0.625
            # row to 1.0, made be_upper < be_lower on losing rows, and inverted the §2
            # stacked chart. Values > 100% read as "impossible by this factor".
            d["be_upper"]    = (d.be_lower * 1.6).round(3)
            d = d.drop(columns=["_in_t","_out_t"])

            view = (d[["model","option_id","gpu_class","rental_term","listed_rate",
                       "mixed_tps_per_gpu","revenue_ph","profit_ph","margin_pct",
                       "be_lower","be_upper","price_src","perf_src"]]
                    .sort_values("profit_ph", ascending=False))
            _styler7 = (view.style
                    .format({"listed_rate":"${:,.2f}",
                             "mixed_tps_per_gpu":"{:,.0f}",
                             "revenue_ph":"${:,.3f}","profit_ph":"${:,.3f}",
                             "be_lower":"{:.1%}","be_upper":"{:.1%}",
                             "margin_pct":"{:.1f}%"},
                            na_rep="—")
                    .set_properties(**{"text-align":"right"})
                    .set_properties(subset=["model","option_id","price_src","perf_src"],
                                    **{"text-align":"left"})
                    .set_properties(subset=["option_id"],**{"min-width":"280px","max-width":"340px",
                                                             "white-space":"normal","word-break":"break-word"})
                    .set_caption(f"all {len(view):,} rows (scroll)  ·  mix={mix}  ·  utilisation={UTIL:.0%}"))
            _push(W.HTML(f'<div style="max-height:400px; overflow-y:auto; border:1px solid #ccc; border-radius:4px;">{_styler7.to_html()}</div>'))

            # ── Suggested token prices per class for this workload ──────────
            # OpenRouter / provider pricing is by token class (fresh input,
            # cached-input, output), so we split the cost-per-GPU·h across
            # those three classes and report a per-class breakeven and
            # gross-margin suggestion, next to each class's public list price.
            #
            # Cost-share weights (heuristic — matches typical inference econ):
            #   weight_fresh  = 1.0  prefill = full attention scan; dominant work
            #   weight_cached = 0.2  cache-hit path only looks up; ~5x cheaper
            #   weight_out    = 1.0  decode = autoregressive per generated token
            #
            # For each GPU we pick the offer that minimises $/h (already in the
            # top-N table). Split that $/h by class weighted-cost-share, divide
            # by that class's tokens/h, get per-class $/1M at each target GM.
            # be_* columns are emphasized (bold) in the styled view; p30/p50/p70 give
            # the price ladder at 30/50/70 % target gross margin.
            # (constants _SUG_MARGIN_TIERS, _MIX_FRACTION, _CLASS_WEIGHT now live at cell top)

            # ── workload-aware throughput anchors (mirror §1 cell 5 logic) ──
            # These exist in cell 7 too so that pricing responds to the
            # workload slider (isl/osl/kv). Keep the dict in sync with §1.
            _PUBLIC_PRICE_LIST = {
                'glm-5.3-flash':       dict(fresh=0.15,  cached=0.03,  out=0.50,
                                            ref='OpenRouter GLM-5.2 listing'),
                'kimi-k3':             dict(fresh=3.00,  cached=0.30,  out=15.00,
                                            ref='Moonshot official rate card'),
                'deepseek-v4.1-flash': dict(fresh=0.15,  cached=0.003, out=0.60,
                                            ref='DeepSeek Day-6 report / OpenRouter'),
                'hy4-preview':         dict(fresh=0.80,  cached=0.038, out=2.48,
                                            ref='OpenRouter Hy4 preview listing'),
                'mimo-v2.6-flash':     dict(fresh=0.14,  cached=0.003, out=0.28,
                                            ref='eesel.ai V2.6 table / OpenRouter (2026-10-28 check: input raised $0.10→$0.14)'),
            }
            # workload sliders (isl, osl, kv) come from the c7_w_work dropdown
            work_isl, work_osl, work_kv = isl, osl, kv   # already unpacked at top of c7_run
            sel_model = c7_w_model.value
            if sel_model not in ('__all__', COMPARE):
                sel_rows = d[d.throughput_case=='best']
                sugg_lines = []
                for gpu_cls, gsub in sel_rows.groupby('gpu_class'):
                    best = gsub.loc[gsub.profit_ph.idxmax()] if len(gsub) else None
                    if best is None: continue
                    hourly_cost = float(best.eff_cost_ph)
                    # workload-aware throughput: b300 anchor × per-model hw_scale
                    _in_b300, _out_b300, _win_gain = _b300_anchors_for(sel_model, work_isl, work_osl, work_kv)
                    _hw = _HW_SCALE_LOCAL[sel_model].get(gpu_cls, 0.0)
                    _in_t  = _in_b300  * _hw
                    _out_t = _out_b300 * _hw
                    total_tps = _in_t + _out_t
                    tps_cl = {c: total_tps * _MIX_FRACTION[c] for c in _MIX_FRACTION}
                    wsum = sum(_MIX_FRACTION[c] * _CLASS_WEIGHT[c] for c in _MIX_FRACTION)
                    row = dict(gpu=gpu_cls.upper(),
                               offer=str(best.option_id)[:30],
                               cost_h=hourly_cost,
                               tps_total=total_tps)
                    for c, mfrac in _MIX_FRACTION.items():
                        share    = (mfrac * _CLASS_WEIGHT[c]) / wsum
                        cost_c   = hourly_cost * share
                        # utilisation scales tokens sold per hour; cost stays fixed
                        if c == "fresh":
                            toks_h = _in_t * (1 - work_kv/100.0) * 3600 * UTIL
                        elif c == "cached":
                            toks_h = _in_t * (work_kv/100.0) * 3600 * UTIL
                        else:
                            toks_h = _out_t * 3600 * UTIL
                        be_c     = cost_c / max(toks_h, 1e-9) * 1e6
                        row[f'be_{c}'] = be_c
                        for tier, gm in _SUG_MARGIN_TIERS:
                            row[f'p{tier}_{c}'] = be_c / (1 - gm)
                        pub_c = _PUBLIC_PRICE_LIST[sel_model][c]
                        row[f'pub_{c}'] = pub_c
                    sugg_lines.append(row)
                sugg = pd.DataFrame(sugg_lines).sort_values('be_fresh').reset_index(drop=True)
                if len(sugg):
                    pub = _PUBLIC_PRICE_LIST[sel_model]
                    pub_blend = sum(pub[c] * _MIX_FRACTION[c] for c in _MIX_FRACTION)
                    _hdr = ('<div style="font:600 12.5px/1.5 system-ui; color:#0f172a; margin:10px 0 4px;">'
                            f'Suggested token prices per class — {sel_model} · mix={mix} · util={UTIL:.0%}</div>'
                            '<div style="font:400 11px/1.5 system-ui; color:#475569; margin:0 0 2px;">'
                            f"public list (from §1): fresh=${pub['fresh']:g} · cached=${pub['cached']:g} · out=${pub['out']:g} · blended=${pub_blend:.3f}/1M  [{pub['ref']}]<br>"
                            'cost-share weights: fresh=1.0 · cached=0.20 · out=1.0  (heuristic — tune with measured backend split)<br>'
                            'columns: <b>be_*</b> = breakeven $/1M  ·  p30/p50/p70_* = price at 30/50/70 % GM  ·  pub_* = public list $/1M</div>')
                    _push(W.HTML(_hdr))
                    show_cols = ['gpu','offer','cost_h','tps_total']
                    for c in ('fresh','cached','out'):
                        show_cols += [f'be_{c}', f'p30_{c}', f'p50_{c}', f'p70_{c}', f'pub_{c}']
                    show = sugg[[c for c in show_cols if c in sugg.columns]]
                    _fmt = show.copy()
                    for col in _fmt.columns:
                        if col == 'tps_total':
                            _fmt[col] = _fmt[col].map(lambda v: f'{v:,.0f}')
                        elif col.startswith('be_'):
                            _fmt[col] = _fmt[col].map(lambda v: f'**${v:,.4f}**' if abs(v) < 1 else f'**${v:,.2f}**')
                        elif col not in ('gpu','offer'):
                            _fmt[col] = _fmt[col].map(lambda v: f'${v:,.4f}' if abs(v) < 1 else f'${v:,.2f}')

                    fmt_dict = {'cost_h':'${:,.2f}', 'tps_total':'{:,.0f}'}
                    for c in ('fresh','cached','out'):
                        fmt_dict[f'be_{c}']  = '${:,.4f}'
                        fmt_dict[f'p30_{c}'] = '${:,.4f}'
                        fmt_dict[f'p50_{c}'] = '${:,.4f}'
                        fmt_dict[f'p70_{c}'] = '${:,.4f}'
                        fmt_dict[f'pub_{c}'] = '${:,.3f}'
                    _be_cols = [c for c in show.columns if c.startswith('be_')]
                    _sg_show = (show.style
                            .format(fmt_dict)
                            .set_properties(subset=_be_cols,
                                            **{'font-weight':'bold',
                                               'background-color':'#fff3cd'})
                            .set_caption(f'{sel_model} · per-class token prices — bold/amber = breakeven (be_*), surrounding columns are the 30/50/70 % margin price ladder; pub_* = the market\'s list price'))
                    _push(W.HTML(_sg_show.to_html()))
            if c7_w_model.value == "__all__":
                per_model = (d.groupby("model")
                              .agg(median_prof=("profit_ph","median"),
                                   median_margin_pct=("margin_pct","median"),
                                   best_offer=("profit_ph","max"))
                              .round(2).reset_index())
                _pm_styler = (per_model.style
                        .format({"median_prof":"${:,.2f}",
                                 "best_offer":"${:,.2f}",
                                 "median_margin_pct":"{:.1f}%"})
                        .set_caption("per-model rollup"))
                _push(W.HTML(_pm_styler.to_html()))
            else:
                dd = d[(d.mix=="agentic") & (d.throughput_case=="best") & d.gpu_class.isin(["b300","gb300"])]
                if len(dd):
                    dd = dd.sort_values("be_lower").head(20)
                    fig, ax = plt.subplots(figsize=(12, 3.8))
                    x = list(range(len(dd)))
                    cols = ['#2a9d8f' if b <= 1.0 else '#e9c46a' for b in dd.be_lower]
                    ax.bar(x, dd.be_lower, color=cols, label='be_lower · linear')
                    ax.bar(x, (dd.be_upper - dd.be_lower).fillna(0),
                            bottom=dd.be_lower, color='#e76f51', alpha=.35,
                            label='be_upper − linear')
                    ax.axhline(1.0, color='#1d3557', lw=1.5, ls='--', label='break-even (100% busy)')
                    ax.set_xticks(x,
                                  [s[:24] + " @ $" + f"{lr:.2f}" + "/h" for s, lr in zip(dd.option_id, dd.listed_rate)],
                                  rotation=70, ha='right', fontsize=7)
                    ax.set_ylabel("share of GPU time that must be busy to break even")
                    ax.set_title(f"Breakeven utilisation — {c7_w_model.value} · my B300/GB300 offers")
                    ax.legend(fontsize=8); ax.grid(axis='y', alpha=.3)
                    plt.tight_layout()
                    _img_buf = io.BytesIO()
                    fig.savefig(_img_buf, format='png', bbox_inches='tight')
                    plt.close(fig)
                    _push(W.Image(value=_img_buf.getvalue(), format='png',
                                  layout=W.Layout(max_width='960px')))

        c7_output.children = tuple(_new_children)

    for _w in (c7_w_model, c7_w_price, c7_w_gpu, c7_w_work, c7_w_util):
        _w.observe(c7_run, "value")

    display(W.VBox([
        W.HBox([c7_w_model, c7_w_price]),
        W.HBox([c7_w_gpu, c7_w_work, c7_w_util]),
        c7_output
    ]))
    c7_run()



## A1 · Appendix · GPU prices — my quotes × public market

Two side-by-side tables (no merging), with `eff_cost_ph` (listed_rate + upfront fee amortised over the contractual term) and `breakeven_tps_sheet` (the minimum workload throughput per GPU per second required for the listed price to break even).


In [37]:
from IPython.display import display

mine = my_quotes.assign(price_src="my_quotes")
pub  = pub_prices.assign(price_src="public_market")

mine['delivery_rfs'] = mine.get('delivery_rfs', '')
mine['availability_signal'] = mine.get('availability_signal', '')
pub1 = pub.copy()
pub1['delivery_rfs'] = ''
pub1['availability_signal'] = ''
keep = ['option_id','gpu_class','rental_term','listed_rate','upfront',
        'delivery_rfs','availability_signal',
        'upfront_pct','price_date','source_url']
mine['_price_src'] = 'my_quotes'
pub1['_price_src'] = 'public_market'
allp = pd.concat([mine[keep+['_price_src']], pub1[keep+['_price_src']]], ignore_index=True, sort=False)


GPU_ORDER = ['gb300', 'b300', 'gb200', 'b200', 'h200', 'h100', 'mi355x', 'mi325x']
for gpu in GPU_ORDER:
    sub = allp[allp.gpu_class == gpu].sort_values(['listed_rate']).reset_index(drop=True)
    if not len(sub):
        continue
    n_my = (sub['_price_src']=='my_quotes').sum()
    n_pub = (sub['_price_src']=='public_market').sum()
    colors = sub['_price_src'].map({'my_quotes':'#fff4d6','public_market':'#e8f1fc'})
    styler = (sub.drop(columns=['_price_src']).style
              .format({'listed_rate':'${:,.2f}','upfront':'${:,.0f}','upfront_pct':'{:,.0f}%'})
              .set_caption(f'**{gpu.upper()}** · {n_my} my + {n_pub} public · sorted by listed rate'))
    # apply row-color with a single vectorised .apply (one pass, no loop) —
    # row-by-row set_properties calls rebuild the CSS rule tree each time and
    # hang the kernel for ~1k rows.
    _n_cols = sub.shape[1] - 1   # styler was built on sub.drop(_price_src)
    _my_rows = set(sub.index[sub['_price_src'] == 'my_quotes'])
    styler = styler.apply(
        lambda row: ['background-color: #fff4d6'] * _n_cols
                    if row.name in _my_rows
                    else ['background-color: #e8f1fc'] * _n_cols,
        axis=1)
    display(styler)

,option_id,gpu_class,rental_term,listed_rate,upfront,delivery_rfs,availability_signal,upfront_pct,price_date,source_url
0,GB300-3year-Verda-GB300-36M/50%--@-$-3.95-Marc,gb300,3_year,$3.95,"$51,903",March 2027,Coming Inventory,50%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
1,GB300-3year-Verda-GB300-36M/25%--@-$-4.2-March,gb300,3_year,$4.20,"$27,594",March 2027,Coming Inventory,25%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
2,GB300-3year-YOTTA-GB300--—-36M-@-$4.8,gb300,3_year,$4.80,"$63,072","Ideally delivery by December, but if things don’t go smoothly, by February next year at the latest.",Current / Monitor,50%,2026-10-16,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
3,GB300-3year-Ornn-—-GB300-NVL72-—-US-East-—-864,gb300,3_year,$4.85,"$31,864","Contract service window: January 25, 2027 through January 25, 2030.",Coming Inventory,25%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
4,GB300-5year-Compute-Exchange-—-GB300-NVL72-—-U-2,gb300,5_year,$4.94,$nan,"US Central; go-live January 2027; up to 5,184 GB300 NVL72 GPUs. Listing may be splittable.",Coming Inventory,nan%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
5,GB300-3year-hosted.ai-GB300-MA--36M/30%--@-$-4,gb300,3_year,$4.95,$nan,January 2027 target stated 14–18 week lead time,Current / Monitor,nan%,2026-09-24,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
6,GB300-3year-YOTTA-GB300--—-36M-@-$5.00,gb300,3_year,$5.00,"$39,420","Ideally delivery by December, but if things don’t go smoothly, by February next year at the latest.",Current / Monitor,30%,2026-10-16,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
7,GB300-3year-Turboscale-GB300-—-36M-/-20%-@-$5.,gb300,3_year,$5.10,"$26,806",Feb 2027,Current / Monitor,20%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
8,GB300-5year-vast.ai-GB300-Ohio-60M-@-$5.10--pa,gb300,5_year,$5.10,"$46,910","Contractual committed RFS: February 1, 2027, per Dmitry on October 2. Supplier says the allocation requires a fast decision.",Current / Monitor,21%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
9,GB300-5year-Compute-Exchange-—-GB300-NVL72-—-U,gb300,5_year,$5.10,$nan,"US South; go-live March 2027; up to 3,488 GB300 NVL72 GPUs. Listing may be splittable.",Coming Inventory,nan%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559


,option_id,gpu_class,rental_term,listed_rate,upfront,delivery_rfs,availability_signal,upfront_pct,price_date,source_url
0,B300-CoreWeave-Spot,b300,spot,$4.48,$0,,,0%,2026-10-22,https://www.coreweave.com/pricing
1,B300-3year-hosted.ai-—-B300-Dallas-—-36M-@-$4,b300,3_year,$4.60,$nan,"Dallas, Texas; January 2027 deployment. Netpreme asked whether the table RFS is January 5; provider has not yet confirmed the exact date.",Current / Monitor,nan%,2026-10-26,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
2,B300-3year-Clarity-—-B300-Canada-—-8-nodes-—-,b300,3_year,$4.74,"$24,913","4 nodes RFS December 1, 2026; 4 additional nodes RFS January 15, 2027; same fabric.",Coming Inventory,20%,2026-10-26,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
3,B300-3year-hosted.ai-—-B300-Dallas-—-32-nodes,b300,3_year,$4.75,$nan,"Dallas, Texas; 32 nodes / 256 GPUs; RFS February 15, 2027.",Coming Inventory,nan%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
4,B300-3year-Clarity-—-B300-Canada-—-8-nodes-—--2,b300,3_year,$4.78,"$25,124","4 nodes RFS December 1, 2026; 4 additional nodes RFS January 15, 2027; same fabric.",Coming Inventory,20%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
5,B300-3year-Ornn-—-B300-Sweden-—-12-nodes-—-36,b300,3_year,$4.80,"$31,536","Conservative start January 1, 2027; provider is optimistic about partial or full December delivery.",Coming Inventory,25%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
6,B300-1year-Clarity-—-B300-Canada-—-1-node-—-1,b300,1_year,$4.97,$nan,1 node available end of November 2026; expansion of 1–2 more nodes indicated shortly.,Coming Inventory,nan%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
7,B300-2year-hosted.ai-—-B300-Dallas-—-32-nodes,b300,2_year,$5.50,"$28,908","Dallas, Texas; 32 nodes / 256 GPUs; RFS February 15, 2027.",Coming Inventory,30%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
8,B300-ondemand-DeepInfra-B300-On-demand-—-$5.89-e,b300,on_demand,$5.89,$nan,nan,Current / Monitor,nan%,2026-10-21,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
9,"B300-ondemand-DigitalOcean-B300-Spot-—-Oct-23,-2",b300,on_demand,$6.17,$nan,Spot capacity; availability varies by region and idle capacity.,Current / Monitor,nan%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559


,option_id,gpu_class,rental_term,listed_rate,upfront,delivery_rfs,availability_signal,upfront_pct,price_date,source_url
0,GB200-GMI-OD,gb200,on_demand,$8.00,$0,,,0%,2026-10-22,https://www.gmicloud.ai/en/pricing
1,GB200-CoreWeave-OD,gb200,on_demand,$10.50,$0,,,0%,2026-10-22,https://www.coreweave.com/pricing
2,GB200-AWS-CapacityBlock,gb200,reserved,$10.58,$nan,,,100%,2026-10-22,https://aws.amazon.com/ec2/capacityblocks/pricing


,option_id,gpu_class,rental_term,listed_rate,upfront,delivery_rfs,availability_signal,upfront_pct,price_date,source_url
0,B200-CoreWeave-Spot,b200,spot,$4.26,$0,,,0%,2026-10-22,https://www.coreweave.com/pricing
1,B200-GCP-A4-Res,b200,reserved,$4.28,$nan,,,100%,2026-10-22,https://www.spheron.network/blog/google-cloud-a4-b200-pricing-2026-cost-vs-spheron
2,B200-ondemand-DeepInfra-B200-On-demand-—-$4.89-e,b200,on_demand,$4.89,$nan,nan,Current / Monitor,nan%,2026-10-21,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
3,B200-GMI-OD,b200,on_demand,$5.00,$0,,,0%,2026-10-22,https://www.gmicloud.ai/en/pricing
4,B200-ondemand-RunPod-B200-Community-Cloud-Live-S,b200,on_demand,$5.98,$nan,On-demand Pods across RunPod marketplace regions; actual regional stock must be checked at deployment.,Current / Monitor,nan%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
5,B200-Hyperstack-OD,b200,on_demand,$6.00,$0,,,0%,2026-10-22,https://www.hyperstack.cloud/pricing
6,B200-Modal-OD,b200,on_demand,$6.25,$0,,,0%,2026-10-22,https://guptadeepak.com/tools/top-7-gpu-cloud-ai-compute-providers-2026
7,B200-Together-Res,b200,reserved,$6.79,$nan,,,100%,2026-10-22,https://docs.together.ai/docs/gpu-clusters-billing
8,B200-Lambda-OD,b200,on_demand,$6.99,$0,,,0%,2026-10-22,https://checkthat.ai/brands/lambda-labs/pricing
9,B200-RunPod-SecOD,b200,on_demand,$7.99,$0,,,0%,2026-10-22,https://www.runpod.io/pricing


,option_id,gpu_class,rental_term,listed_rate,upfront,delivery_rfs,availability_signal,upfront_pct,price_date,source_url
0,H200-GMI-OD,h200,on_demand,$3.20,$0,,,0%,2026-10-22,https://www.gmicloud.ai/en/pricing
1,H200-1year-Compute-Exchange-—-H200-South-Afri,h200,1_year,$3.30,"$28,908","South Africa; at least Tier 3; virtualized with InfiniBand passthrough. Provider said RFS next week or the following as of Oct 23 (approximately Oct 26–Nov 8, 2026).",Current / Monitor,100%,2026-10-26,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
2,H200-1year-Compute-Exchange-—-H200-South-Afri-2,h200,1_year,$3.40,"$29,784","South Africa; at least Tier 3; virtualized with InfiniBand passthrough. Provider said RFS next week or the following as of Oct 23 (approximately Oct 26–Nov 8, 2026).",Current / Monitor,100%,2026-10-26,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
3,H200-1year-Compute-Exchange-—-H200-South-Afri-4,h200,1_year,$3.40,"$14,892","South Africa; at least Tier 3; virtualized with InfiniBand passthrough. Provider said RFS next week or the following as of Oct 23 (approximately Oct 26–Nov 8, 2026).",Current / Monitor,50%,2026-10-26,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
4,H200-ondemand-DeepInfra-H200-On-demand-—-$3.49-e,h200,on_demand,$3.49,$nan,nan,Current / Monitor,nan%,2026-10-21,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
5,H200-ondemand-Hyperbolic-H200-Live-Marketplace-R,h200,on_demand,$3.49,$nan,Marketplace/on-demand capacity; actual provider inventory and price fluctuate and require dashboard confirmation.,Current / Monitor,nan%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
6,H200-1year-Compute-Exchange-—-H200-South-Afri-3,h200,1_year,$3.50,"$15,330","South Africa; at least Tier 3; virtualized with InfiniBand passthrough. Provider said RFS next week or the following as of Oct 23 (approximately Oct 26–Nov 8, 2026).",Current / Monitor,50%,2026-10-26,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
7,H200-ondemand-RunPod-H200-Community-Cloud-Live-S,h200,on_demand,$3.59,$nan,On-demand Pods across RunPod marketplace regions; actual regional stock must be checked at deployment.,Current / Monitor,nan%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
8,H200-Hyperstack-OD,h200,on_demand,$3.99,$0,,,0%,2026-10-22,https://www.hyperstack.cloud/pricing
9,H200-Together-Res,h200,reserved,$3.99,$nan,,,100%,2026-10-22,https://www.together.ai/gpu-clusters


,option_id,gpu_class,rental_term,listed_rate,upfront,delivery_rfs,availability_signal,upfront_pct,price_date,source_url
0,H100-ondemand-GMI-H100-—-1M-renewal-@-$1.80-(Sep,h100,on_demand,$1.80,$nan,Existing deployed H100 capacity; current $1.80 rate explicitly honored through end-October 2026,Current / Monitor,nan%,2026-09-30,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
1,H100-ondemand-Vutlr-H100,h100,on_demand,$2.25,$nan,nan,Current / Monitor,nan%,2026-09-22,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
2,H100-ondemand-Exabits-H100-VM,h100,on_demand,$2.40,$nan,nan,Current / Monitor,nan%,2026-09-22,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
3,H100-1year-PhysionLabs-4-6-H100-nodes,h100,1_year,$2.40,$nan,nan,Current / Monitor,nan%,2026-08-10,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
4,H100-1year-andromeda-H100-US-1Y,h100,1_year,$2.60,"$6,833",nan,Current / Monitor,30%,2026-07-27,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
5,H100-ondemand-RunPod-H100-Community-Cloud-—-SXM-,h100,on_demand,$2.69,$nan,On-demand Pods across RunPod marketplace regions; actual regional stock must be checked at deployment.,Current / Monitor,nan%,2026-10-23,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
6,H100-ondemand-MassedCompute-H100,h100,on_demand,$2.73,$nan,nan,Current / Monitor,nan%,2026-09-22,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
7,H100-ondemand-Hyperbolic-H100-Live-Marketplace-R,h100,on_demand,$2.89,$nan,"Marketplace alert repeatedly matched the requested H100 filter of 8+ GPUs on October 22–23, 2026; exact provider, region and live price require dashboard confirmation.",Current / Monitor,nan%,2026-10-26,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
8,H100-ondemand-DeepInfra-H100-On-demand-—-$2.90-e,h100,on_demand,$2.90,$nan,nan,Current / Monitor,nan%,2026-10-21,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559
9,H100-1year-andromeda-H100-US-West-1Y,h100,1_year,$2.96,"$5,186",nan,Current / Monitor,20%,2026-07-27,https://app.notion.com/GPU-Inventory-Tracker-7d381c47df9183b18e65811c04d05559


,option_id,gpu_class,rental_term,listed_rate,upfront,delivery_rfs,availability_signal,upfront_pct,price_date,source_url
0,MI355X-Vultr-preemptible,mi355x,preemptible,$2.59,$0,,,0%,2026-10-22,https://www.spheron.network/blog/amd-mi300x-mi355x-pricing-2026
1,MI355X-TensorWave-quote,mi355x,on_demand,$2.95,$0,,,0%,2026-10-22,https://www.spheron.network/blog/amd-mi300x-mi355x-pricing-2026
2,MI355X-Crusoe-OD,mi355x,on_demand,$7.70,$0,,,0%,2026-10-22,https://www.crusoe.ai/cloud/pricing
3,MI355X-OCI-OD,mi355x,on_demand,$8.60,$0,,,0%,2026-10-22,https://computeprices.com/compare/nebius-vs-oracle


## A2 · Appendix · My benchmark × InferenceX

Per-(ISL, KV, OSL) ceiling stars come from `glm53_b300_25k_13k_benchmark_raw.csv` (182 valid runs, SLA-passing only). For each (ISL, KV, OSL) cell the plot shows the single best run by `out_tps_per_gpu`.

* x-axis: P50 per-user decode (tok/s/user) — what clients feel.
* y-axis: output tok/s per GPU — what you actually sell.
* Stars are labelled `kv<N>·osl<M>` so the legs of the trajectory are readable.
* Orange boxes are InferenceX public anchors at the same axes (per-GPU t/s on agentic-traces, fp4-mtp).


In [38]:
from IPython.display import display, HTML

# ── Same plot, same axes — me vs InferenceX ─────────────────────────────────
# Visual grammar (3 tiers):
#   MY B300 lab    → CIRCLE  (●)   teal solid = SLA-pass   ·   coral hollow = SLA-fail
#   MY ceiling     → STAR    (★)   dark navy = isl=25k     ·   medium    = isl=13k
#   InferenceX     → SQUARE  (■)   orange    = B300-class  ·   gray      = other GPUs
fig, ax = plt.subplots(figsize=(14, 5.5))

# (1) my B300 lab dots, all circles — color encodes SLA.
# Gateway rows (tp == 'na') are NOT lab dots — they're a hosted service, so
# they get their own marker in (1b) and the legend.
_lab = my_bench[pd.to_numeric(my_bench.tp, errors='coerce').notna()].copy()
for tp, tp_sub in _lab.groupby('tp'):
    met = tp_sub[tp_sub.meets_gates == 'yes']
    bad = tp_sub[tp_sub.meets_gates != 'yes']
    ax.scatter(met.decode_med_tps_user, met.out_tput_per_gpu,
               c='#2a9d8f', marker='o', s=54, alpha=.92, edgecolors='#fff', linewidths=.6, zorder=3)
    if len(bad):
        ax.scatter(bad.decode_med_tps_user, bad.out_tput_per_gpu,
                   facecolors='none', edgecolors='#e76f51', marker='o',
                   s=64, alpha=.85, linewidths=1.4, zorder=3)

# (1b) hosted-gateway measurement(s) — hosted service, not a GPU.
_gw = my_bench[pd.to_numeric(my_bench.tp, errors='coerce').isna()].copy()
if len(_gw):
    ax.scatter(_gw.decode_med_tps_user, _gw.out_tput_per_gpu,
               marker='D', s=110, c='#5b3ea8', edgecolors='#fff', linewidths=1.2, zorder=6)
    for _, r in _gw.iterrows():
        ax.annotate(f"gateway c{int(r.concurrency)} osl{int(r.osl)}",
                    xy=(r.decode_med_tps_user, r.out_tput_per_gpu),
                    xytext=(6, -12), textcoords='offset points',
                    fontsize=7.5, color='#5b3ea8', alpha=.95)
    if len(bad):
        ax.scatter(bad.decode_med_tps_user, bad.out_tput_per_gpu,
                   facecolors='none', edgecolors='#e76f51', marker='o',
                   s=64, alpha=.85, linewidths=1.4, zorder=3)

# (2) per-(isl, kv, osl) ceiling — keep the per-osl best only
ceil = None
for base in ['.', 'data', 'grid', 'data_measured']:
    pth = os.path.join(base, 'glm53_b300_25k_13k_benchmark_raw.csv')
    if os.path.exists(pth):
        rb = pd.read_csv(pth)
        rb = rb[rb['valid'].astype(str).str.upper() == 'TRUE']
        rb = rb[rb['pass_gmi_sla'].astype(str).str.upper() == 'TRUE']
        ceil = (rb.sort_values('out_tps_per_gpu', ascending=False)
                  .drop_duplicates(['isl','cache_hit_rate','osl'], keep='first')
                  .rename(columns={'cache_hit_rate':'kv',
                                   'out_tps_per_gpu':'out_tps',
                                   'decode_p50':'decode_tps'})
                  .loc[:, ['isl','kv','osl','conc','out_tps','decode_tps']])
        ceil_src = 'raw_lab (25k/13k benchmark)'
        break
if ceil is None:
    mb = my_bench[my_bench.meets_gates == 'yes'].copy()
    ceil = (mb.sort_values('out_tput_per_gpu', ascending=False)
              .drop_duplicates(['osl'], keep='first')
              .assign(isl=0, kv=0)
              .rename(columns={'out_tput_per_gpu':'out_tps',
                               'decode_med_tps_user':'decode_tps',
                               'concurrency':'conc'})
              .loc[:, ['isl','kv','osl','conc','out_tps','decode_tps']])
    ceil_src = 'my_bench (summary only)'

isl_color = {25000: '#1d3557', 13000: '#588157', 0: '#4a4e69'}
isl_size  = {25000: 480,       13000: 400,       0: 340}

for isl, g in ceil.groupby('isl'):
    ax.scatter(g.decode_tps, g.out_tps,
               marker='*', s=isl_size.get(isl, 340),
               c=isl_color.get(isl, '#4a4e69'),
               edgecolors='#fff', linewidths=1.1, alpha=.95, zorder=5)
    for _, r in g.iterrows():
        ann = f"kv{int(r.kv)}·osl{int(r.osl)}" if r.kv else f"osl{int(r.osl)}"
        ax.annotate(ann, xy=(r.decode_tps, r.out_tps),
                    xytext=(6, -10), textcoords='offset points',
                    fontsize=7, color='#444', alpha=.9)

# (3) InferenceX anchors — all squares, B300-class hot color, others gray
if infx_glm53 is not None:
    g2 = infx_glm53[infx_glm53['model']=='glm-5.3'].sort_values('tput_per_gpu')
    for _, r in g2.iterrows():
        is_b300 = r.gpu_class in ('b300','gb300')
        ax.scatter(r.med_intvty_tps_user, r.tput_per_gpu,
                   marker='s', s=110,
                   c='#f4a261' if is_b300 else '#8c99a6',
                   edgecolors='#a03e00' if is_b300 else '#46505b',
                   linewidths=1.4, alpha=.95, zorder=4)
        ax.annotate(r.gpu_class.upper(), xy=(r.med_intvty_tps_user, r.tput_per_gpu),
                    xytext=(4, 2), textcoords='offset points',
                    fontsize=8, color='#222')

ax.set_xlabel('P50 per-user decode (tok/s/user)  [x = what clients feel]')
ax.set_ylabel('Output tok/s per GPU  [y = what you sell]')
ax.set_title(f'MY B300 lab vs public InferenceX anchors  ·  ceiling = per-(isl,kv,osl) best  ·  src: {ceil_src}')
ax.set_xlim(0, 450)
ax.set_ylim(0, 1700)
ax.grid(alpha=.3)

# 3-entry legend — one per visual tier
from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0],[0], marker='o', color='none', markerfacecolor='#2a9d8f',
           markeredgecolor='#fff', markersize=9, label='MY B300 · SLA-pass'),
    Line2D([0],[0], marker='o', color='none', markerfacecolor='none',
           markeredgecolor='#e76f51', markersize=9, markeredgewidth=1.4, label='MY B300 · SLA-fail'),
    Line2D([0],[0], marker='*', color='none', markerfacecolor='#1d3557',
           markeredgecolor='#fff', markersize=16, label='MY B300 ceiling · isl=25k'),
    Line2D([0],[0], marker='*', color='none', markerfacecolor='#588157',
           markeredgecolor='#fff', markersize=16, label='MY B300 ceiling · isl=13k'),
    Line2D([0],[0], marker='s', color='none', markerfacecolor='#f4a261',
           markeredgecolor='#a03e00', markersize=9, label='InferenceX B300-class (public)'),
    Line2D([0],[0], marker='s', color='none', markerfacecolor='#8c99a6',
           markeredgecolor='#46505b', markersize=9, label='InferenceX other GPUs (public)'),
    Line2D([0],[0], marker='D', color='none', markerfacecolor='#5b3ea8',
           markeredgecolor='#fff', markersize=9, label='Netpreme prod gateway (hosted svc)'),
]
ax.legend(handles=legend_handles, fontsize=8, ncol=3, loc='lower right',
          framealpha=.95, edgecolor='#bbb', handletextpad=0.4, columnspacing=1.0)

plt.tight_layout(); plt.show()

# ── Auditable ceiling table ──────────────────────────────────────────────────
print(f'ceiling runs per (isl, kv, osl)  ·  source: {ceil_src}')
print(ceil.sort_values(['isl','kv','osl']).round(1).to_string(index=False))
print()
best = (my_bench[my_bench.meets_gates == 'yes']
        .sort_values(['osl', 'out_tput_per_gpu']))



ceiling runs per (isl, kv, osl)  ·  source: my_bench (summary only)
 isl  kv  osl  conc  out_tps  decode_tps
   0   0  128    32    132.0        40.0
   0   0  256     8    215.0        66.0
   0   0  512    32    348.0        72.0
   0   0 1024    32    523.0        98.0
   0   0 2048    24   1033.5       104.0



/var/folders/n9/mk3c339x6bb4dbvpb3_yyx780000gn/T/ipykernel_46479/1278233143.py:102: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## A3 · Appendix · InferenceX reference — full frontier (10 models × 11 GPUs)

Bubble plot of every (model, GPU) anchor in `infx_all_models_anchors.csv` for the `agentic-traces` sequence. Bubble size ∝ concurrency; per-SKU labels only appear on the top-3 models by `out_tput_per_gpu`. The right pane is the zoom table for the user's B300 family.

Below the frontier figure, the same dataframe is also pivoted to print the **Kimi-K3 per-GPU hw_scale** the live calculator uses (the calculator source of truth).


In [39]:
from IPython.display import display

if infx_all_models is None:
    print('infx_all_models_anchors.csv not bundled — run fetch_infx_all_models.py')
else:
    ag = infx_all_models[infx_all_models.sequence=='agentic-traces'].copy()
    ag = ag.sort_values('run_date').drop_duplicates(['model','gpu'], keep='last')

    mdisp = {'glm5.2':'GLM-5.2 (5.3-Flash L4)','kimik3':'Kimi-K3','dsv4':'DeepSeek-V4-Pro',
             'dsv41flash':'DSV4.1-Flash','glm5.1':'GLM-5.1','minimaxm3':'MiniMax-M3',
             'minimaxm31flash':'MiniMax-M3.1-Flash','musespark':'Muse-Spark-Open',
             'qwen3.5':'Qwen-3.5-397B-A17B','qwen3.8next':'Qwen3.8-Flash-Next'}
    cmap_map = {
        'glm5.2':'#e76f51',        # terracotta — the model you actually care about
        'kimik3':'#9b5de5',        # purple
        'dsv41flash':'#2a9d8f',
        'dsv4':'#94d2bd',
        'minimaxm3':'#fcbf49',
        'minimaxm31flash':'#f77f00',
        'musespark':'#a8dadc',
        'qwen3.5':'#4361ee',
        'qwen3.8next':'#3a0ca3',
        'glm5.1':'#bdbdbd',
    }

    # ── prune annotations: per SKU keep ONLY glm5.2 + the top-2 non-glm5.2 by throughput ──
    anno_keep = set()
    for g, sub in ag.groupby('gpu'):
        sub = sub.sort_values('out_tput_per_gpu', ascending=False)
        glm52_row = sub[sub.model=='glm5.2']
        if len(glm52_row):
            anno_keep.add(tuple(glm52_row.iloc[0][['model','gpu']]))
        for _, r in sub[sub.model!='glm5.2'].head(2).iterrows():
            anno_keep.add((r['model'], r['gpu']))

    def _sz(c): return (20 + (float(c)**0.45) * 9)

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(16, 7.4), gridspec_kw=dict(width_ratios=[1.15, 1]))

    # zoom panel: hide unlabeled points entirely (annotated ones only) for clarity
    zoom_models = {'glm5.2','dsv41flash','dsv4','kimik3','minimaxm31flash'}   # B300-class leaders

    def _draw(ax, models=None, gpu_filter=None, show_anno=True):
        for m in mdisp:
            if models and m not in models: continue
            sub = ag[ag.model==m]
            if gpu_filter is not None:
                sub = sub[sub.gpu.isin(gpu_filter)]
            if not len(sub): continue
            szs = sub.conc.map(_sz)
            alpha_each = .80 if m in ('glm5.2','kimik3') else .55
            ax.scatter(sub.intivity_tps_user.clip(lower=0.1),
                       sub.out_tput_per_gpu.clip(lower=0.1),
                       s=szs, c=cmap_map[m], alpha=alpha_each, edgecolors='#222', linewidths=.4,
                       label=mdisp[m] if ax is axL else None, zorder=3)
            if show_anno:
                for _, r in sub.iterrows():
                    if (r.model, r.gpu) not in anno_keep: continue
                    ax.annotate(r.gpu.upper(),
                                (max(r.intivity_tps_user,.1), max(r.out_tput_per_gpu,.1)),
                                fontsize=6.6, ha='center', va='center', color='#fff',
                                weight='bold', zorder=4,
                                bbox=dict(boxstyle='round,pad=0.12', fc='none', ec='none', alpha=0),
                                )

    _draw(axL)
    axL.set_xscale('log'); axL.set_yscale('log')
    axL.set_xlim(0.3, 2000); axL.set_ylim(0.5, 10000)
    axL.set_xlabel('per-user decode, tok/s per user  [log]')
    axL.set_ylabel('output tok/s per GPU  [log]')
    axL.set_title('all 10 models × all 11 SKUs  ·  only top-3-per-SKU labelled')
    axL.grid(alpha=.25, which='both')

    # my B300 lab — sit on both panels
    my_row = my_bench[(my_bench.meets_gates=='yes') & (my_bench.tp==2) & (my_bench.osl==2048) & (my_bench.concurrency==24)].iloc[0]
    for ax_ in (axL, axR):
        ax_.scatter(my_row.decode_med_tps_user, my_row.out_tput_per_gpu, s=550, c='#ffd60a', marker='s',
                    edgecolors='#111', linewidths=1.6, zorder=6, label='MY B300 lab · TP2·osl 2048·c=24·KV60%' if ax_ is axL else None)
        glm52_b300 = ag[(ag.model=='glm5.2') & (ag.gpu=='b300')].out_tput_per_gpu.iloc[0]
        ax_.axhline(glm52_b300, ls='--', lw=1.3, color='#c1121f', alpha=.65, zorder=2)

    axR.annotate('MY B300\n1,034 t/s', (my_row.decode_med_tps_user, my_row.out_tput_per_gpu),
                 xytext=(6,-8), textcoords='offset points', fontsize=8.5, weight='bold', color='#111',
                 bbox=dict(boxstyle='round,pad=0.25', fc='#fff3bf', ec='#111', lw=.5), zorder=7)

    # right panel: TABLE of anchors around B300 / GB300 / GB200
    axR.axis('off')
    zoom_df = ag[(ag.gpu.isin(['b300','gb300','gb200']))].copy()
    zoom_df['lbl'] = zoom_df.model.map(mdisp) + ' · ' + zoom_df.gpu.str.upper()
    zoom_df = zoom_df.sort_values('out_tput_per_gpu', ascending=False).head(12)
    rows = []
    my_row_lbl = f"MY B300 lab  TP2·osl2048·c24·KV60%  {my_row.out_tput_per_gpu:,.0f} t/s/GPU"
    for i, (_, r) in enumerate(zoom_df.iterrows()):
        col = cmap_map[r.model]
        rows.append((i,
                     f"{mdisp[r.model]:<26s} {r.gpu.upper():<6s} {r.out_tput_per_gpu:>5.0f} t/s/GPU @ c={r.conc:.0f}",
                     col))
    y0 = 0.97; dy = 0.075
    axR.set_xlim(0, 1); axR.set_ylim(0, 1)
    axR.text(0.02, y0 - (-1)*dy, f"Top anchors on B300 / GB300 / GB200  (agentic-traces, fp4-mtp)",
             fontsize=9.5, weight='bold', family='monospace', transform=axR.transAxes)
    for k, txt, col in rows:
        y = y0 - (k+1)*dy
        axR.add_patch(plt.Rectangle((0.0, y-dy/2-0.005), 1.0, dy*0.92, facecolor=col, alpha=0.18, edgecolor='none'))
        axR.text(0.03, y, txt, fontsize=8.6, family='monospace', va='center', color='#111')
    y_my = y0 - (len(rows)+1)*dy
    axR.add_patch(plt.Rectangle((0.0, y_my-dy/2-0.005), 1.0, dy*0.92, facecolor='#ffd60a', alpha=0.85, edgecolor='#111', lw=.5))
    axR.text(0.03, y_my, my_row_lbl, fontsize=8.6, family='monospace', va='center', weight='bold', color='#111')
    axR.set_title('zoom: anchors on your B300 family  ·  sorted by t/s', loc='left', fontsize=10)

    axL.axhline(glm52_b300, ls='--', lw=1.3, color='#c1121f', alpha=.65, zorder=2)

    fig.suptitle('InferenceX frontier fleet (agentic-traces, 2026-10-26)  ·  bubble ∝ conc  ·  yellow = my B300 lab',
                 fontsize=11, y=1.0)
    handles, labels = axL.get_legend_handles_labels()
    fig.legend(handles, labels, fontsize=7.5, ncol=6, loc='lower center', bbox_to_anchor=(0.5,-0.03), frameon=True)
    plt.tight_layout(rect=[0,0.04,1,0.97])
    plt.show()

    # table after the plot
    P = ag.pivot_table(index='model', columns='gpu', values='out_tput_per_gpu', aggfunc='max')
    P = P.reindex(sorted(P.index, key=lambda m: -P.loc[m].max()))
    display(P.round(0).style
              .format("{:.0f}", na_rep="·")
              .background_gradient(cmap='viridis', axis=None)
              .set_caption("out tok/s per GPU  ·  agentic-traces  ·  bestPerSku frontier"))
    # ── Kimi-K3 hw_scale sub-table ────────────────────────────────────────────────
    # derived from the same infx_all_models dataframe (newer schema: 'gpu', 'out_tput_per_gpu')
    kimi_ag = ag[ag.model=='kimik3'].copy()
    b300_ref = kimi_ag[kimi_ag.gpu=='b300'].out_tput_per_gpu.iloc[0]
    kimi_ag['hw_scale'] = (kimi_ag.out_tput_per_gpu / b300_ref).round(3)
    show_kimi = (kimi_ag[['gpu','out_tput_per_gpu','conc','intivity_tps_user','precision','framework','run_date','source_url','hw_scale']]
                 .sort_values('hw_scale', ascending=False))
    show_kimi.columns = ['gpu_class','out_tps/GPU','conc','decode_tps/user','precision','framework','run_date','source','hw_scale (vs B300=1)']
    display(show_kimi.style
            .format({'out_tps/GPU':'{:,.1f}','decode_tps/user':'{:,.1f}','hw_scale (vs B300=1)':'{:.3f}'})
            .set_caption("Kimi-K3 per-GPU anchors · derived from the same agentic-traces frame"))




/var/folders/n9/mk3c339x6bb4dbvpb3_yyx780000gn/T/ipykernel_46479/1989862560.py:118: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


gpu,b200,b300,gb200,gb300,h100,h200,mi300x,mi325x,mi355x,tpuv7,vr200
model,,,,,,,,,,,
dsv41flash,2503,3262,2534,7528,123,445,182,282,1340,·,·
qwen3.8next,638,706,·,1298,72,297,·,·,·,·,2542
dsv4,50,683,27,1077,·,48,·,·,649,·,2112
minimaxm3,351,491,276,346,·,·,·,·,448,·,1243
qwen3.5,376,568,·,972,·,·,·,·,654,361,·
minimaxm31flash,628,902,632,803,45,77,·,·,436,·,·
glm5.2,128,298,312,37,·,·,·,·,139,·,858
musespark,298,309,248,384,·,82,·,·,253,·,·
kimik3,87,88,76,173,·,1,·,·,75,53,313


,gpu_class,out_tps/GPU,conc,decode_tps/user,precision,framework,run_date,source,hw_scale (vs B300=1)
23,vr200,312.7,256,102.4,fp4,dynamo-vllm,2026-10-23,https://inferencex.semianalysis.com/inference/kimi-k3,3.571
24,gb300,173.5,240,55.4,fp4,dynamo-vllm,2026-10-23,https://inferencex.semianalysis.com/inference/kimi-k3,1.981
26,b300,87.6,70,7.9,fp4,vllm,2026-08-16,https://inferencex.semianalysis.com/inference/kimi-k3,1.000
27,b200,86.6,72,16.3,fp4,dynamo-vllm,2026-10-26,https://inferencex.semianalysis.com/inference/kimi-k3,0.988
25,gb200,75.9,56,24.6,fp4,dynamo-vllm,2026-10-21,https://inferencex.semianalysis.com/inference/kimi-k3,0.867
28,mi355x,74.7,48,46.7,fp4,vllm-disagg,2026-10-14,https://inferencex.semianalysis.com/inference/kimi-k3,0.852
30,tpuv7,53.1,72,25.1,fp4,vllm,2026-10-17,https://inferencex.semianalysis.com/inference/kimi-k3,0.606
29,h200,1.1,26,1.5,fp4,vllm,2026-08-07,https://inferencex.semianalysis.com/inference/kimi-k3,0.012


## A4 · Appendix · OpenRouter weekly top-5 models, grouped by week

For each week Oct 2025 → Oct 2026, the top-5 OpenRouter models by token volume (in billions of tokens). Pulled from `openrouter_weekly_top9_10-2025_to_10-2026.csv`.

In [40]:
if or_weekly is None:
    print('openrouter_weekly_top9_10-2025_to_10-2026.csv not bundled — run the fetch step first')
else:
    from IPython.display import display
    ork = or_weekly.set_index('model')

    def month_top5(yearmonth, cols):
        rows = []
        for w in cols:
            s = ork[w].drop(index=['Others', 'Total'], errors='ignore').sort_values(ascending=False)
            top5 = s.head(5)
            rows.append({'week of': w,
                         **{f'#{i+1}': (f'{top5.index[i]} ({top5.iloc[i]:.1f}T)' if i < len(top5) else '—')
                            for i in range(5)}})
        df = pd.DataFrame(rows).set_index('week of')
        return (df.style.set_caption(f'**{yearmonth}**')
                .set_table_styles([{'selector': 'caption', 'props':
                    [('font-weight','bold'),('font-size','13px'),('text-align','left'),('padding','6px 0 2px 0')]}]))

    months = {}
    for w in ork.columns:
        months.setdefault(w[:7], []).append(w)
    for ym in sorted(months):
        display(month_top5(ym, months[ym]))

,#1,#2,#3,#4,#5
week of,,,,,
2025-10-20,grok-code-fast-1 (0.7T),claude-4.5-sonnet (0.3T),gemini-2.5-flash (0.2T),gemini-2.5-pro (0.1T),grok-4-fast (0.1T)
2025-10-27,grok-code-fast-1 (1.5T),claude-4.5-sonnet (0.7T),gemini-2.5-flash (0.3T),minimax-m2 (free) (0.2T),gemini-2.5-pro (0.2T)


,#1,#2,#3,#4,#5
week of,,,,,
2025-11-03,grok-code-fast-1 (1.4T),claude-4.5-sonnet (0.7T),gemini-2.5-flash (0.4T),minimax-m2 (free) (0.3T),grok-4-fast (0.2T)
2025-11-10,grok-code-fast-1 (1.5T),claude-4.5-sonnet (0.6T),gemini-2.5-flash (0.4T),minimax-m2 (0.2T),grok-4-fast (0.2T)
2025-11-17,grok-code-fast-1 (1.2T),claude-4.5-sonnet (0.5T),grok-4.1-fast (0.4T),gemini-2.5-flash (0.4T),minimax-m2 (0.2T)
2025-11-24,grok-4.1-fast (free) (1.6T),grok-code-fast-1 (1.0T),claude-4.5-sonnet (0.4T),gemini-2.5-flash (0.4T),grok-4.1-fast (0.2T)


,#1,#2,#3,#4,#5
week of,,,,,
2025-12-01,grok-code-fast-1 (0.9T),grok-4.1-fast (free) (0.6T),claude-4.5-sonnet (0.4T),gemini-2.5-flash (0.4T),claude-4.5-opus (0.2T)
2025-12-08,grok-code-fast-1 (0.6T),gemini-2.5-flash (0.5T),claude-4.5-sonnet (0.4T),gpt-oss-120b (0.3T),claude-4.5-opus (0.2T)
2025-12-15,grok-code-fast-1 (0.5T),gemini-2.5-flash (0.4T),claude-4.5-sonnet (0.4T),gpt-oss-120b (0.2T),claude-4.5-opus (0.2T)
2025-12-22,grok-code-fast-1 (0.5T),mimo-v2-flash-20251210 (free) (0.4T),gemini-2.5-flash (0.4T),claude-4.5-sonnet (0.4T),deepseek-v3.2 (0.3T)
2025-12-29,grok-code-fast-1 (0.4T),deepseek-v3.2 (0.4T),claude-4.5-sonnet (0.4T),gemini-2.5-flash (0.4T),mimo-v2-flash-20251210 (free) (0.4T)


,#1,#2,#3,#4,#5
week of,,,,,
2026-01-05,claude-4.5-sonnet (0.5T),grok-code-fast-1 (0.4T),mimo-v2-flash-20251210 (free) (0.4T),gemini-3-flash-preview (0.4T),claude-4.5-opus (0.4T)
2026-01-12,claude-4.5-opus (0.6T),claude-4.5-sonnet (0.6T),mimo-v2-flash-20251210 (free) (0.5T),grok-code-fast-1 (0.5T),gemini-3-flash-preview (0.4T)
2026-01-19,claude-4.5-sonnet (0.7T),mimo-v2-flash-20251210 (free) (0.6T),grok-code-fast-1 (0.5T),gemini-3-flash-preview (0.5T),deepseek-v3.2 (0.4T)
2026-01-26,claude-4.5-sonnet (0.8T),gemini-3-flash-preview (0.7T),deepseek-v3.2 (0.5T),grok-code-fast-1 (0.4T),gemini-2.5-flash (0.4T)


,#1,#2,#3,#4,#5
week of,,,,,
2026-02-02,kimi-k2.5-0127 (1.2T),gemini-3-flash-preview (0.8T),claude-4.5-sonnet (0.7T),deepseek-v3.2 (0.7T),minimax-m2.1 (0.4T)
2026-02-09,minimax-m2.5 (1.4T),kimi-k2.5-0127 (1.3T),gemini-3-flash-preview (0.8T),deepseek-v3.2 (0.8T),glm-5 (0.6T)
2026-02-16,minimax-m2.5 (2.6T),kimi-k2.5-0127 (1.0T),gemini-3-flash-preview (0.9T),glm-5 (0.8T),deepseek-v3.2 (0.7T)
2026-02-23,minimax-m2.5 (1.6T),gemini-3-flash-preview (1.0T),deepseek-v3.2 (0.8T),kimi-k2.5-0127 (0.7T),claude-4.6-sonnet (0.6T)


,#1,#2,#3,#4,#5
week of,,,,,
2026-03-02,minimax-m2.5 (1.9T),gemini-3-flash-preview (1.0T),deepseek-v3.2 (0.8T),claude-4.6-opus (0.8T),step-3.5-flash (free) (0.7T)
2026-03-09,minimax-m2.5 (1.7T),step-3.5-flash (free) (1.3T),deepseek-v3.2 (1.0T),gemini-3-flash-preview (1.0T),claude-4.6-sonnet (0.9T)
2026-03-16,mimo-v2-pro (1.5T),step-3.5-flash (free) (1.5T),minimax-m2.5 (1.3T),deepseek-v3.2 (1.1T),claude-4.6-sonnet (1.0T)
2026-03-23,mimo-v2-pro (3.9T),step-3.5-flash (free) (1.5T),minimax-m2.7 (1.3T),deepseek-v3.2 (1.2T),claude-4.6-sonnet (1.0T)
2026-03-30,qwen3.6-plus-04-02 (free) (4.6T),mimo-v2-pro (3.1T),qwen3.6-plus-preview (free) (1.6T),step-3.5-flash (free) (1.3T),minimax-m2.7 (1.2T)


... (3 outputs left out)

,#1,#2,#3,#4,#5
week of,,,,,
2026-07-06,hy3-20260706 (free) (6.1T),mimo-v2.5 (5.9T),deepseek-v4-flash (5.2T),minimax-m3 (4.3T),glm-5.2 (3.2T)
2026-07-13,hy3-20260706 (free) (11.5T),mimo-v2.5 (9.3T),deepseek-v4-flash (5.4T),minimax-m3 (3.8T),glm-5.2 (3.7T)
2026-07-20,mimo-v2.5 (10.5T),deepseek-v4-flash (6.4T),hy3 (3.9T),glm-5.2 (3.3T),deepseek-v4-pro (3.2T)
2026-07-27,deepseek-v4-flash (7.2T),mimo-v2.5 (6.3T),hy3 (4.8T),deepseek-v4-pro (3.3T),glm-5.2 (2.9T)


,#1,#2,#3,#4,#5
week of,,,,,
2026-08-03,deepseek-v4-flash ·0731 (8.8T),hy3 (8.1T),deepseek-v4-flash (5.9T),mimo-v2.5 (5.4T),gpt-5.6-luna (4.4T)
2026-08-10,deepseek-v4-flash ·0731 (11.2T),hy3 (10.0T),gpt-5.6-luna (5.3T),deepseek-v4-flash (4.8T),glm-5.2 (4.3T)
2026-08-17,deepseek-v4-flash ·0731 (11.6T),ox-alpha (11.6T),mimo-v2.5 (9.9T),hy3 (8.2T),deepseek-v4-flash (5.5T)
2026-08-24,ox-alpha (15.7T),deepseek-v4-flash ·0731 (12.3T),mimo-v2.5 (9.1T),gpt-5.6-luna (7.8T),hy3 (6.7T)
2026-08-31,hy4-preview (14.7T),gpt-5.6-luna (12.9T),glm-5.3-flash (12.4T),deepseek-v4-flash ·0731 (12.4T),deepseek-v4-flash (5.2T)


,#1,#2,#3,#4,#5
week of,,,,,
2026-09-07,gpt-5.6-luna (18.2T),hy4-preview (16.8T),glm-5.3-flash (11.9T),deepseek-v4-flash ·0731 (11.6T),mimo-v2.5 (7.8T)
2026-09-14,deepseek-v4.1-flash (15.8T),glm-5.3-flash (14.1T),hy4-preview (12.5T),gpt-5.6-luna (9.7T),deepseek-v4-flash ·0731 (9.4T)
2026-09-21,deepseek-v4.1-flash (19.6T),glm-5.3-flash (16.3T),space-bunny-alpha (13.9T),hy4-preview (9.6T),gpt-5.6-luna (8.5T)
2026-09-28,space-bunny-alpha (38.7T),deepseek-v4.1-flash (25.6T),glm-5.3-flash (9.7T),mimo-v2.6-flash (9.6T),hy4-preview (6.5T)


,#1,#2,#3,#4,#5
week of,,,,,
2026-10-05,deepseek-v4.1-flash (41.0T),glm-5.3-flash (12.2T),mimo-v2.6-flash (11.5T),step-5-preview (8.8T),hy4-preview (8.3T)
2026-10-12,deepseek-v4.1-flash (32.8T),glm-5.3-flash (12.2T),step-5-preview (11.5T),mimo-v2.6-flash (9.9T),hy4-preview (7.6T)
2026-10-19,deepseek-v4.1-flash ·1017 (17.6T),minimax-m3.1-flash-20261019 (free) (13.7T),deepseek-v4.1-flash (11.1T),glm-5.3-flash (6.5T),hy4-preview (5.6T)


---
### Data sources
| class | rows come from | tag | access date |
|---|---|---|---|
| your offers | your Notion quotes table, link per row in `my_gpu_opts.csv` | `price_src=my_quotes` | 2026-10-22 |
| your benchmark | GLM-5.3-Flash benchmark.pdf (Oct-5/7; 2×/4×B300, vLLM+MTP5) | `perf_src=my_benchmark` | 2026-10-05/07 |
| market prices | 16-provider web scan, per-row URL + upfront_kind in `public_gpu_prices.csv`; negative results in `public_prices_notes.md` | `price_src=public_market` | 2026-10-22 |
| public perf | InferenceX GLM-5.3 / DeepSeek-V4-Flash / Kimi-K3 runs, per-row URL in `infx_glm53_anchors.csv` | `perf_src=inferencex` | 2026-10-22 |
| OpenRouter top-5 | live page + official State-of-AI/Inferred reports + dated 3rd-party snapshots, per-row URL + confidence (V/R/N/I) in `openrouter_top5_by_month.csv` | — | 2026-10-22 |

**Upfront semantics** — `fee` = extra on top of hourly (amortised into `eff_cost_ph`); `prepayment` = part of the contract value or a usage-credit, `eff_cost_ph = listed_rate`; `none` = pure on-demand / monthly-billed commit. AWS Capacity Blocks and GCP A4 future-reservations are classified **prepayment** under this rule (pay-in-full, nothing metered afterwards); Together AI reserved and Hyperstack on-demand are also `prepayment` (credits); every other public row is `none`. No public `fee` row exists in the 2026-10-22 scan.